# **Classification: Airline Passenger Referral Prediction EDA**



##### **Project Type**    - Binary Classification
##### **Contribution**    - Individual
##### **Name**            - Soham Kar

# **Project Summary -**

Data includes airline reviews from 2006 to 2019 for popular airlines around the world with multiple choice and free text questions. Data is scraped in Spring 2019. The main objective is to predict whether passengers will refer the airline to their friends.

# **GitHub Link -**

Provide your GitHub Link here.

# **Problem Statement**


**The main objective is to predict(classify yes or no) whether passengers will refer the airline to their friends based on one's own experience. Binary Classification task**

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

In [ ]:
#install packages
!pip install missingno

### Import Libraries

In [ ]:
# Import Libraries
import pandas as pd
import geopandas as gpd
import plotly.express as px
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import requests
from io import StringIO
import io
import missingno as msno
import re
%matplotlib inline

### Dataset Loading

In [ ]:
# Load Dataset
'''
DOWNLOAD THE DATASET
'''
def load_data_set():
  # the csv file is hosted in a GoogleDrive for ease of access by anyone
  link = 'https://drive.google.com/file/d/1CQBrCN_APJQ0ClZ4CpiHaXRws-vxT7hU/view?usp=sharing'
  # Extract file ID from the link(for google drive files)
  file_id = link.split("/")[-2]
  # Construct the download link for the file
  download_link = f"https://drive.google.com/uc?id={file_id}"
  # Make a request to download the file
  response = requests.get(download_link)
  # Check if the request was successful
  if response.status_code == 200:
    csv_content = StringIO(response.text)
    df = pd.read_csv(csv_content)
    return df
  else:
    print(f"Failed to download the file. Status Code: {response.status_code}")
    return None

# Call the function to load the dataset and store it in df_load
df_load = load_data_set()

In [ ]:
# Remove extra rows(rows with all attributes as NaNs)
df_original = df_load.dropna(how = "all")
df_original = df_original.reset_index(inplace = False, drop=True)

### Dataset First View

In [ ]:
# Dataset First Look
df_original.head()

In [ ]:
df_original.tail()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
# Dataset Rows & Columns count
print(f"Dataset rows:    {df_original.shape[0]}")
print(f"Dataset Columns: {df_original.shape[1]}")

### Dataset Information

In [ ]:
# Dataset Info
df_original.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
# Checking duplicated rows
def check_duplicate_(df):
  print(f"Original dataset size:                             {df.shape[0]}")
  print(f"Duplicated airline:                                {df.airline.duplicated().sum()}")
  print(f"Duplicated author:                                 {df.author.duplicated().sum()}")
  print(f"Duplicated review_date:                            {df.review_date.duplicated().sum()}")
  print(f"Duplicated customer_review:                        {df.customer_review.duplicated().sum()}")
  # dropping duplicates where all the variables have the same value
  df_all_duplicates = df[df.duplicated(subset=['airline',
                                               'overall',
                                               'author',
                                               'review_date',
                                               'customer_review',
                                               'aircraft',
                                               'traveller_type',
                                               'cabin',
                                               'route',
                                               'date_flown',
                                               'seat_comfort',
                                               'cabin_service',
                                               'food_bev',
                                               'entertainment',
                                               'ground_service',
                                               'value_for_money',
                                               'recommended'], keep='first')]

  print(f"Duplicate rows(all attribute/variable value same): {df_all_duplicates.shape[0]}")
  # dropping duplicates expect the first ocuurance
  df_new = df.drop_duplicates(keep="first")
  # reset index
  df_new = df_new.reset_index(inplace = False, drop=True)
  print(f"\nNew dataset size:                                  {df_new.shape[0]}")
  print(f"Data Reduction:                                    {((df.shape[0] - df_new.shape[0])/df.shape[0])*100:.2f} % (less than 10 %)")
  return df_new

df_no_Duplicates = check_duplicate_(df_original)

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
df_no_Duplicates.isnull().sum()

In [ ]:
# Visualizing the missing values
plt.figure(figsize=(5, 5))
# Generate the missing data matrix
msno.matrix(df_no_Duplicates)
# Show the plot
plt.show()

### What did you know about your dataset?

The original dataset is not well-prepared for further analysis, as it contains 131895 rows and 17 features. There are extra rows i.e. blank rows where every feature has a NaN value, which has been dropped **dropna(how = "all")**. Most of the features are either objects or floats. If necessary, it needs to be converted into the required datatype. After the necessary cleaning, the dataset will be ready for preprocessing steps, allowing the focus to be on feature engineering and model development to achieve accurate predictions.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Describe for object data types
df_no_Duplicates[['airline',
                  'author',
                  'aircraft',
                  'traveller_type',
                  'cabin',
                  'route',
                  'recommended']].describe(include='object')

In [ ]:
df_no_Duplicates.describe()

### Variables Description

* **airline:** Name of the airline.
* **overall:** Overall point is given to the trip between 1 to 10.
* **author:** Author of the trip
* **review date:** Date of the Review
* **customer review:** Review of the customers in free text format
* **aircraft:** Type of the aircraft
* **traveller type:** Type of traveler (e.g. business, leisure)
* **cabin:** Cabin at the flight date flown: Flight date
* **seat comfort:** Rated between 1-5
* **cabin service:** Rated between 1-5
* **foodbev:** Rated between 1-5
* **entertainment:** Rated between 1-5
* **ground service:** Rated between 1-5
* **value for money:** Rated between 1-5
* **recommended:** Binary, target variable.

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for object type variable.
for col in df_no_Duplicates[['airline',
                             'author',
                             'aircraft',
                             'traveller_type',
                             'cabin']].columns:
  print(f"Unique values present in {col}: {df_no_Duplicates[col].nunique()}")

In [ ]:
for col in df_no_Duplicates.columns:
  print(f"Unique values present in {col}: {df_no_Duplicates[col].nunique()}")

## ***3. Data Wrangling***

### Data Wrangling Code

In [ ]:
def review_date_obj(df):
  # convert review_date to datetime obj
  df['review_date'] = pd.to_datetime(df['review_date'], errors='coerce', format='mixed')
  df['year'] = pd.to_datetime(df['review_date']).dt.year
  # remove 'date_flown' column
  df = df.drop('date_flown', axis=1, errors='ignore')
  return df

df_date_clean = review_date_obj(df_no_Duplicates)

In [ ]:
def customer_review_transform(df):
  # Remove additional characters from the reviews
  def change_review(txt):
    review_regex = re.compile(
        r"(É\?\? Trip Verified \| [A-Za-z\s]+ to [A-Za-z\s]+(?: via [A-Za-z\s]+)?[.,])|"
        r"(Not Verified \| [A-Za-z\s]+ to [A-Za-z\s]+(?: via [A-Za-z\s]+)?[.,])|"
        r"([A-Za-z\s]+ to [A-Za-z\s]+[.,])|"
        r"(É\?\? Trip Verified \|)"
    )
    return review_regex.sub('', txt).strip()
  # Apply the regex transformation
  df['customer_review'] = df['customer_review'].apply(change_review)
  return df

df_cust = customer_review_transform(df_date_clean)

In [ ]:
def create_to_from_via(df):
  # compile the regular expression once outside the function
  route_regex = re.compile(r'(.*) to (.*?)(?: via (.*))?$')
  # function to extract from, to, and via based on regex groups
  def extract_route_parts(txt):
    regex_match = route_regex.search(txt)
    if regex_match:
      from_ = regex_match.group(1).strip().lower()
      to_ = regex_match.group(2).strip().lower()
      via_ = regex_match.group(3).strip().lower() if regex_match.group(3) else None
      return from_, via_, to_
  # create new columns from existing 'route' column
  df[['from', 'via', 'to']] = df['route'].astype(str).apply(lambda x: pd.Series(extract_route_parts(x)))
  df = df.drop('route', axis=1)
  return df

df_from_via_to = create_to_from_via(df_cust)

In [ ]:
# create a test dataframe
def create_new_df(df):
  print(f"Original Dataset size:                                                {df.shape[0]}")
  # dropping all the rows where target variable is None/NaN
  df = df.dropna(subset=['recommended'])
  df = df.reset_index(inplace=False, drop=True)
  print(f"Original Dataset size after dropping null values in 'recommended':    {df.shape[0]}")
  df_ = df.drop(['airline', 'author', 'review_date', 'aircraft', 'from', 'via', 'to', 'customer_review', 'cabin'], axis=1)
  nulls = df_[df_.isnull().sum(axis=1) > 7]
  df_unseen = df.reindex(nulls.index)
  df_unseen.reset_index(inplace=False, drop=True)
  print(f"Unseen Dataset size:                                                  {df_unseen.shape[0]}")
  df_1 = df.drop(index=nulls.index)
  df_1 = df_1.reset_index(inplace=False, drop=True)
  print(f"Work Dataset size:                                                    {df_1.shape[0]}")

  return df_1, df_unseen

"""
The create_new_df func. we will extract two datasets:
1. df_clean: will be used for data viz, transformation, training, and testing.
2. df_test_valid: will be used as an *unseen* dataset to check the performance of the final model. This dataset contains
                  all the rows where most of the column data is NaN, except the target variable!
"""
df_clean, df_test_valid = create_new_df(df_from_via_to)

In [ ]:
df_clean.tail()

In [ ]:
df_test_valid.tail()

### What all manipulations have you done and insights you found?

* **review_date_obj(df)**, processes the review_date column in a DataFrame and performs two main tasks:

  * **Convert review_date to a DateTime object**

    * **df['year'] = pd.to_datetime(df['review_date']).dt.year:**
    Extracts the year from review_date and stores it in a new column called "year".
    * **df['review_date'] = pd.to_datetime(df['review_date'], format='mixed'):**
    Converts the review_date column into a proper datetime format.
    Remove the date_flown column

  * **df = df.drop('date_flown', axis=1):**
  Deletes the date_flown column from the DataFrame.
Return the cleaned DataFrame

  The modified DataFrame is returned and stored as df_date_clean.

* **customer_review_transform(df)** cleans the customer_review column by removing certain patterns using regular expressions (regex).

    * **Defines change_review(txt)**

      Uses a regex pattern to find and remove specific text patterns from customer reviews.
      The regex removes:
      * "É?? Trip Verified | ..."
      * "Not Verified | ..."
      * "City to City."
      
      The modified text is returned without these extra parts.
      Applies change_review() to the customer_review column

  df['customer_review'] = df['customer_review'].apply(change_review)
  Every row's review text gets cleaned.
  Returns the cleaned DataFrame

* The function **create_to_from_via(df)** extracts "from," "to," and "via" locations from the "route" column and creates three new columns: "from", "to", and "via". It then removes the original "route" column.

* **The function create_new_df(df)** cleans the dataset and splits it into two parts: df_1 (Work Dataset) and df_unseen (Unseen Dataset). It first removes rows where the "recommended"(target variable) column is missing, resets the index, and drops unnecessary columns. Then, it identifies rows with more than 7 missing values and moves them to df_unseen. The remaining rows form df_1, which is used for further analysis. The function prints the dataset sizes at each step and returns the two datasets. This ensures that data quality improves by handling missing values effectively while keeping highly incomplete records separate.



## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### **Chart - 1: Airlines with Highest and Lowest Number of Reviews**


In [ ]:
# Chart - 1 visualization code
df_new = df_clean.groupby(['airline']).count()
# Creating subplots
fig, axes = plt.subplots(ncols=2, figsize=(20, 5))

# Plotting the first subplot of the figure
sns.barplot(df_new.sort_values(by='customer_review', ascending=False).head(15),
            x="customer_review",
            y="airline",
            hue='airline',
            orient='h',
            legend=False,
            palette="pastel",
            edgecolor=".4",
            ax=axes[0])
# Set title
axes[0].set_title('Airlines with Most Number of Reviews(Top 15)',
                  fontweight ='bold')
# Set x-axis
axes[0].set_xlabel('Num. of reviews',
                   fontweight ='bold')
# Set y-axis
axes[0].set_ylabel('Airline',
                   fontweight ='bold')
axes[0].tick_params(axis='x')
for container in axes[0].containers:
    axes[0].bar_label(container)


# Plotting the first subplot of the figure
sns.barplot(df_new.sort_values(by='customer_review', ascending=False).tail(15),
            x="recommended",
            y="airline",
            hue='airline',
            orient='h',
            legend=False,
            palette="pastel",
            edgecolor=".4",
            ax=axes[1])
# Set title
axes[1].set_title('Airlines with Least Number of Reviews(Bottom 15)',
                  fontweight ='bold')
# Set x-axis
axes[1].set_xlabel('Num. of reviews',
                   fontweight ='bold')
# Set y-axis
axes[1].set_ylabel('Airline',
                   fontweight ='bold')
axes[1].tick_params(axis='x')
for container in axes[1].containers:
    axes[1].bar_label(container)

sns.despine()
# Adjust layout for better spacing
plt.tight_layout()

# Display the subplots
plt.show()

##### 1. Why did you pick the specific chart?

This chart was chosen to analyze the number of customer reviews for different airlines. It provides a comparative view of airlines with the **highest and lowest number of reviews**, helping to identify the airlines that receive the most customer engagement.

##### 2. What is/are the insight(s) found from the chart?

* The top 15 airlines with the most reviews indicate airlines that have a strong customer presence and engagement. Spirit Airlines, Amercian Airlines, United Airlines have the most review counts i.e. **we can assume that that customers prefer these airline companies over other due to their services or lower prices**.
* The bottom 15 airlines with the least reviews suggest airlines that may have a lower market presence or lower customer feedback participation. Similarly, Thai Smile Airways, Tunisair, etc. have **lower review count due to biasness in the dataset or due to their low popularity among customers**.
* This data can help determine which airlines are actively engaging customers and receiving feedback, which is crucial for reputation management. **Although, it should be noted that the dataset can be biased as the data collection may have unintentionaly reviewed customers based on one-or-two location where some companies are prefered more as compared to others**.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, these insights can help in several ways:

* Airlines with **high reviews** can leverage their customer engagement for marketing and loyalty programs.
* Airlines with **low reviews** may need to improve their customer outreach, request feedback, and enhance their services to increase engagement.
* Competitors can use this data to analyze the market position and customer perception of various airlines.

#### **Chart - 2: Airlines with Most and Least AVG. Ratings**

In [ ]:
# Chart - 2 visualization code
df_new = df_clean.groupby(["airline"]).agg({"overall":"mean"}).sort_values(by='overall', ascending=True).head(15)
# Creating subplots
fig, axes = plt.subplots(ncols=2, figsize=(20, 5))

# Plotting the first subplot of the figure
sns.barplot(df_new,
            x="overall",
            y="airline",
            hue='airline',
            orient='h',
            legend=False,
            palette="pastel",
            edgecolor=".4",
            ax=axes[0])
# Set title
axes[0].set_title('Top 15 lowest-rated Airlines',
                  fontweight ='bold')
# Set x-axis
axes[0].set_xlabel('Avg. Rating',
                   fontweight ='bold')
# Set y-axis
axes[0].set_ylabel('Airline',
                   fontweight ='bold')
axes[0].tick_params(axis='x')
for container in axes[0].containers:
    axes[0].bar_label(container)


# Plotting the first subplot of the figure
sns.barplot(df_clean.groupby(["airline"]).agg({"overall":"mean"}).sort_values(by='overall', ascending=False).head(15),
            x="overall",
            y="airline",
            hue='airline',
            orient='h',
            legend=False,
            palette="pastel",
            edgecolor=".4",
            ax=axes[1])
# Set title
axes[1].set_title('Top 15 highest-rated Airlines',
                  fontweight ='bold')
# Set x-axis
axes[1].set_xlabel('Avg.Rating',
                   fontweight ='bold')
# Set y-axis
axes[1].set_ylabel('Airline',
                   fontweight ='bold')
axes[1].tick_params(axis='x')
for container in axes[1].containers:
    axes[1].bar_label(container)

sns.despine()
# Adjust layout for better spacing
plt.tight_layout()

# Display the subplots
plt.show()

##### 1. Why did you pick the specific chart?

This chart was chosen to analyze **airline ratings based on customer reviews**. It provides a clear view of the **top 15 lowest-rated** and **top 15 highest-rated** airlines by their average overall rating. This helps in understanding customer satisfaction levels for different airlines.

##### 2. What is/are the insight(s) found from the chart?

* The **first chart** highlights the airlines with the l**owest average ratings**, indicating airlines that may have poor service quality, delays, or customer dissatisfaction.
* The **second chart** highlights the airlines with the **highest average ratings**, showcasing airlines that provide a better customer experience.
* The data allows for benchmarking against competitors and identifying airlines that excel or struggle in customer satisfaction.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, these insights can be valuable:
* Airlines with **low ratings** can analyze specific pain points and improve customer service, punctuality, and onboard experience.
* Airlines with **high ratings** can use their positive reputation for marketing campaigns and customer loyalty programs.
* Airline competitors can **benchmark themselves** against high-rated airlines and adopt best practices to improve their services.

#### **Chart - 3: Variability in AVG. Ratings over Succeeding Years for the Top 6 Airlines**

In [ ]:
# Chart - 3 visualization code
# Creating subplots
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(20, 10))

# Plotting the first subplot of the figure
sns.lineplot(df_clean[df_clean["airline"] == "Garuda Indonesia"].groupby(["year"]).agg({"overall":"mean"}).sort_values(by='year', ascending=True),
             y="overall",
             x="year",
             marker="o",
             ax=axes[0,0]
            )
# Set title
axes[0,0].set_title('Garuda Indonesia',
                fontweight ='bold')
# Set x-axis
axes[0,0].set_xlabel('year',
                fontweight ='bold')
# Set y-axis
axes[0,0].set_ylabel('avg. rating',
                fontweight ='bold')
axes[0,0].tick_params(axis='x')
for container in axes[0,0].containers:
    axes[0,0].bar_label(container)

# Plotting the first subplot of the figure
sns.lineplot(df_clean[df_clean["airline"] == "EVA Air"].groupby(["year"]).agg({"overall":"mean"}).sort_values(by='year', ascending=True),
             y="overall",
             x="year",
             marker="o",
             ax=axes[0,1]
            )
# Set title
axes[0,1].set_title('EVA Air',
                fontweight ='bold')
# Set x-axis
axes[0,1].set_xlabel('year',
                fontweight ='bold')
# Set y-axis
axes[0,1].set_ylabel('avg. rating',
                fontweight ='bold')
axes[0,1].tick_params(axis='x')
for container in axes[0,1].containers:
    axes[0,1].bar_label(container)

# Plotting the first subplot of the figure
sns.lineplot(df_clean[df_clean["airline"] == "Asiana Airlines"].groupby(["year"]).agg({"overall":"mean"}).sort_values(by='year', ascending=True),
             y="overall",
             x="year",
             marker="o",
             ax=axes[0,2])
# Set title
axes[0,2].set_title('Asiana Airlines',
                fontweight ='bold')
# Set x-axis
axes[0,2].set_xlabel('year',
                fontweight ='bold')
# Set y-axis
axes[0,2].set_ylabel('avg. rating',
                fontweight ='bold')
axes[0,2].tick_params(axis='x')
for container in axes[0,2].containers:
    axes[0,2].bar_label(container)


# Plotting the first subplot of the figure
sns.lineplot(df_clean[df_clean["airline"] == "ANA All Nippon Airways"].groupby(["year"]).agg({"overall":"mean"}).sort_values(by='year', ascending=True),
             y="overall",
             x="year",
             marker="o",
             ax=axes[1,0]
           )
# Set title
axes[1,0].set_title('ANA All Nippon Airways',
                fontweight ='bold')
# Set x-axis
axes[1,0].set_xlabel('year',
                fontweight ='bold')
# Set y-axis
axes[1,0].set_ylabel('avg. rating',
                fontweight ='bold')
axes[1,0].tick_params(axis='x')
for container in axes[1,0].containers:
    axes[1,0].bar_label(container)

# Plotting the first subplot of the figure
sns.lineplot(df_clean[df_clean["airline"] == "China Southern Airlines"].groupby(["year"]).agg({"overall":"mean"}).sort_values(by='year', ascending=True),
             y="overall",
             x="year",
             marker="o",
             ax=axes[1,1])
# Set title
axes[1,1].set_title('China Southern Airlines',
                fontweight ='bold')
# Set x-axis
axes[1,1].set_xlabel('year',
                fontweight ='bold')
# Set y-axis
axes[1,1].set_ylabel('avg. rating',
                fontweight ='bold')
axes[1,1].tick_params(axis='x')
for container in axes[1,1].containers:
    axes[1,1].bar_label(container)

# Plotting the first subplot of the figure
sns.lineplot(df_clean[df_clean["airline"] == "Aegean Airlines"].groupby(["year"]).agg({"overall":"mean"}).sort_values(by='year', ascending=True),
             y="overall",
             x="year",
             marker="o",
             ax=axes[1,2])
# Set title
axes[1,2].set_title('Aegean Airlines',
                fontweight ='bold')
# Set x-axis
axes[1,2].set_xlabel('year',
                fontweight ='bold')
# Set y-axis
axes[1,2].set_ylabel('avg. rating',
                fontweight ='bold')
axes[1,2].tick_params(axis='x')
for container in axes[1,2].containers:
    axes[1,2].bar_label(container)


sns.despine()
# Adjust layout for better spacing
plt.tight_layout()

# Display the subplots
plt.show()

##### 1. Why did you pick the specific chart?

This chart was chosen to analyze the yearly trend of average ratings for **top six airlines** with the **highest avg. ratings from graph - 2**. A line plot is ideal for this analysis because it effectively visualizes trends over time, showing how customer satisfaction has evolved for each airline.

##### 2. What is/are the insight(s) found from the chart?

* The **overall rating trends** for each airline highlight whether customer satisfaction has **improved**, **declined**, or **remained stable** over the years.
* Some airlines might show a **downward trend**, indicating potential service declines, while others may show **upward trends**, suggesting improved customer experience.
* Airlines with **high volatility** in ratings might have inconsistent service quality, whereas those with stable trends have likely maintained a steady reputation.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, these insights are valuable for both airline management and competitors:

* Airlines with **declining ratings** can investigate the reasons behind negative feedback and make service improvements.
* **Well-performing airlines** can use their positive trend as a marketing advantage and reinforce customer trust.
* **Industry competitors** can **analyze these** trends to understand market dynamics and adapt their strategies accordingly.

#### **Chart - 4: Author Count, Recommendation, and AVG. Rating given by Top Authors**

In [ ]:
# Chart - 4 visualization code
df_new = df_clean.groupby(['author']).count()
# Creating subplots
fig, axes = plt.subplots(ncols=3, figsize=(20, 9))

# Plotting the first subplot of the figure
sns.barplot(df_new.sort_values(by='airline', ascending=False).head(25),
            y="author",
            x="airline",
            hue="airline",
            orient='h',
            legend=False,
            palette="pastel",
            edgecolor=".4",
            ax=axes[0])
# Set title
axes[0].set_title('Customers with Most Numbers of Reviews',
                  fontweight ='bold')
# Set x-axis
axes[0].set_xlabel('num. of reviews',
                   fontweight ='bold')
# Set y-axis
axes[0].set_ylabel('author',
                   fontweight ='bold')
axes[0].tick_params(axis='x')
for container in axes[0].containers:
    axes[0].bar_label(container)

top_customers = df_clean.groupby(['author'])['airline'].count().reset_index()
top_25 = top_customers.nlargest(25, 'airline')['author']
filtered_df = df_clean[df_clean['author'].isin(top_25)]
# Plotting the first subplot of the figure
sns.barplot(filtered_df.groupby(['author','recommended']).count().sort_values(by='airline', ascending=False).head(50),
            x="airline",
            y="author",
            hue='recommended',
            orient='h',
            legend=True,
            palette="pastel",
            edgecolor=".4",
            ax=axes[1])
# Set title
axes[1].set_title("Most Frequent Customers' Recommendation Count",
                  fontweight ='bold')
# Set x-axis
axes[1].set_xlabel('num. of reviews',
                   fontweight ='bold')
# Set y-axis
axes[1].set_ylabel('author',
                   fontweight ='bold')
axes[1].tick_params(axis='x')
for container in axes[1].containers:
    axes[1].bar_label(container)

# Plotting the first subplot of the figure
sns.barplot(filtered_df.groupby(['author']).agg({"overall":"mean"}).sort_values(by='overall', ascending=False),
            x="overall",
            y="author",
            orient='h',
            ax=axes[2])
# Set title
axes[2].set_title('Avg. Rating given by Most Frequent Customers',
                  fontweight ='bold')
# Set x-axis
axes[2].set_xlabel('avg. rating',
                   fontweight ='bold')
# Set y-axis
axes[2].set_ylabel('author',
                   fontweight ='bold')
axes[2].tick_params(axis='x')
for container in axes[2].containers:
    axes[2].bar_label(container)

sns.despine()
# Adjust layout for better spacing
plt.tight_layout()

# Display the subplots
plt.show()


##### 1. Why did you pick the specific chart?

This chart was chosen to analyze customer review behavior across three different aspects:

* Customers with the **most reviews** (Top 25 reviewers)
* **Most reviewed customers** giving **recommendation count**
* **Most reviewd customers** avg. rating given

By studying frequent reviewers, businesses can identify key trends in customer sentiment, loyalty, and dissatisfaction.

##### 2. What is/are the insight(s) found from the chart?

* Some customers review airlines frequently, which could indicate either loyal customers or highly engaged critics.
* A segment of reviewers consistently provides high recommendations, potentially showing brand advocates who have positive travel experiences.
* Another segment gives low overall ratings, highlighting dissatisfied customers whose concerns might need addressing.

This data can help airlines spot potential biases, fake reviews, or trends among frequent flyers.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, these insights can drive meaningful business actions:

* **Engaging with top reviewers:** Airlines can build loyalty programs or offer incentives to encourage positive word-of-mouth marketing.
* **Addressing negative reviewers' concerns:** Airlines can identify patterns in negative reviews and take corrective measures to improve service.
* **Detecting potential review manipulation:** If certain reviewers consistently rate airlines unusually high or low, further analysis can help verify the authenticity of reviews.

#### **Chart - 5: Overall Rating Distribution and Recommendation based on given Rating**

In [ ]:
# Chart - 5 visualization code
# Creating subplots
fig, axes = plt.subplots(ncols=2, figsize=(15, 4))

# Plotting the first subplot of the figure
sns.barplot(df_clean.groupby("overall").count(),
            y="customer_review",
            x=df_clean.groupby("overall").count().index,
            hue="review_date",
            orient='v',
            legend=False,
            palette="pastel",
            edgecolor=".4",
            ax=axes[0])
# Set title
axes[0].set_title('Count Plot for Overall Ratings given by Customers',
                  fontweight ='bold')
# Set x-axis
axes[0].set_xlabel('overall',
                   fontweight ='bold')
# Set y-axis
axes[0].set_ylabel('No. of reviews',
                   fontweight ='bold')
axes[0].tick_params(axis='x')
for container in axes[0].containers:
    axes[0].bar_label(container)


df_new = df_clean.groupby(["overall", "recommended"]).count()
# Plotting the first subplot of the figure
sns.barplot(df_new,
            y="customer_review",
            x="overall",
            hue='recommended',
            orient='v',
            legend=True,
            palette="pastel",
            edgecolor=".4",
            ax=axes[1])
# Set title
axes[1].set_title('Recommendation according to Overall',
                  fontweight ='bold')
# Set x-axis
axes[1].set_xlabel('overall',
                   fontweight ='bold')
# Set y-axis
axes[1].set_ylabel('no. of reviews',
                   fontweight ='bold')
axes[1].tick_params(axis='x')
for container in axes[1].containers:
    axes[1].bar_label(container)

sns.despine()
# Adjust layout for better spacing
plt.tight_layout()

# Display the subplots
plt.show()

##### 1. Why did you pick the specific chart?

This chart was chosen to analyze the **distribution of overall ratings given by customers** and their **correlation with recommendations**. The first subplot visualizes how often each rating (e.g., 1-10) was given, while the second compares how many of those ratings led to a **"recommendation."**

##### 2. What is/are the insight(s) found from the chart?

* The **distribution of ratings** reveals whether customer sentiment is mostly positive, negative, or neutral.
* The **second chart highlights the relationship between ratings and recommendations**, confirming whether a high rating reliably leads to a recommendation.
* If lower ratings (e.g., 5-6) still have a significant number of recommendations, it suggests factors beyond overall rating influence customer decisions.
* If there's a gap between high ratings and recommendations, airlines may need to investigate **why some satisfied customers still don’t recommend them**.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, these insights can drive customer experience improvements:

* **Understanding negative reviews:** If 1-3 ratings are frequent, airlines can investigate common complaints and improve services.
* **Identifying service gaps:** If customers rate 8+ but don’t recommend, airlines should explore missing factors like pricing, competition, or inconsistent service.
* **Optimizing marketing and loyalty programs:** If recommendations are strong for a specific rating range, airlines can target marketing campaigns accordingly.

#### **Chart - 6: Review Count and Recommendation by Year**

In [ ]:
# Chart - 6 visualization code
# Creating subplots
fig, axes = plt.subplots(ncols=2, figsize=(20, 5))

# Plotting the first subplot of the figure
sns.barplot(df_clean.groupby(df_clean["review_date"].dt.year).count(),
            y="customer_review",
            x=df_clean.groupby(df_clean["review_date"].dt.year).count().index,
            hue="review_date",
            orient='v',
            legend=False,
            palette="pastel",
            edgecolor=".4",
            ax=axes[0])
# Set title
axes[0].set_title('Number of Reviews by Year',
                  fontweight ='bold')
# Set x-axis
axes[0].set_xlabel('year',
                   fontweight ='bold')
# Set y-axis
axes[0].set_ylabel('count',
                   fontweight ='bold')
axes[0].tick_params(axis='x')
for container in axes[0].containers:
    axes[0].bar_label(container)

df_new=df_clean.groupby([df_clean["review_date"].dt.year, "recommended"]).count()
# Plotting the first subplot of the figure
sns.barplot(df_new,
            x=df_new.index.get_level_values(0),
            y="customer_review",
            hue='recommended',
            orient='v',
            legend=True,
            palette="pastel",
            edgecolor=".4",
            ax=axes[1])
# Set title
axes[1].set_title('Number of Recommendation(Yes/No) by Year',
                  fontweight ='bold')
# Set x-axis
axes[1].set_xlabel('year',
                   fontweight ='bold')
# Set y-axis
axes[1].set_ylabel('count',
                   fontweight ='bold')
axes[1].tick_params(axis='x')
for container in axes[1].containers:
    axes[1].bar_label(container)

sns.despine()
# Adjust layout for better spacing
plt.tight_layout()

# Display the subplots
plt.show()

##### 1. Why did you pick the specific chart?

This chart was chosen to analyze trends in **customer reviews over time and understand the trend of recommendations based on the review year**. The first subplot shows how the total number of reviews has changed over the years, while the second examines the proportion of positive vs. negative recommendations within each year.

##### 2. What is/are the insight(s) found from the chart?

* **Review Trends Over Time:** The first plot shows whether the number of airline reviews has increased or decreased over the years, possibly indicating shifts in customer engagement, market trends, or external events (e.g., pandemic effects).
* **Recommendation Trends:** The second plot highlights whether more or fewer customers recommend airlines over time, which can indicate improvements in airline service or a decline in customer satisfaction.
* If a **decline in reviews is observed**, it could mean fewer customers are leaving feedback, or airline services have changed.
* A drop in **recommendations despite stable review numbers could suggest worsening service or increased competition**.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, these insights help airlines track service perception and customer engagement:

* **If positive recommendations decline**, it signals the need for service improvements.
* **If total reviews drop**, marketing efforts might be required to encourage more customer feedback.

Comparing different years helps airlines pinpoint when service improvements worked or when customer satisfaction dipped.

#### **Chart - 7: Most Common Aircraft Models used by Airlines**

In [ ]:
# Chart - 7 visualization code
# Creating subplots
fig, axes = plt.subplots(ncols=1, figsize=(20, 7))

# Plotting the first subplot of the figure
sns.barplot(df_clean.groupby("aircraft").count().sort_values(by='overall', ascending=False).head(30),
            x="overall",
            y=df_clean.groupby("aircraft").count().sort_values(by='review_date', ascending=False).head(30).index,
            hue="review_date",
            orient='h',
            legend=False,
            palette="pastel",
            edgecolor=".4",
            ax=axes)
# Set title
axes.set_title('Most Common Aircraft Models used by Airlines',
                  fontweight ='bold')
# Set x-axis
axes.set_xlabel('Count',
                   fontweight ='bold')
# Set y-axis
axes.set_ylabel('Model',
                   fontweight ='bold')
axes.tick_params(axis='x')
for container in axes.containers:
    axes.bar_label(container)

sns.despine()
# Adjust layout for better spacing
plt.tight_layout()

# Display the subplots
plt.show()

##### 1. Why did you pick the specific chart?

This chart was selected to highlight the **most commonly used aircraft models based on customer reviews**. It provides insights into which aircraft models airlines rely on the most, showing their popularity or availability across different carriers.

##### 2. What is/are the insight(s) found from the chart?

The most reviewed aircraft models likely indicate the most frequently used ones.
* A **high number of reviews** could suggest widespread usage, while **fewer reviews** may indicate that certain aircraft models are less common or newer.
* If reviews for some **models are significantly higher**, it might indicate that these aircraft are used on popular routes or by major airlines.
* **If older aircraft models still appear frequently**, airlines might be slow in fleet modernization, which could impact customer experience.

#### **Chart - 8: Traveller type Count and Recommendation based on Traveller type**

In [ ]:
# Chart - 8 visualization code
# Creating subplots
fig, axes = plt.subplots(ncols=2, figsize=(20, 4))

# Plotting the first subplot of the figure
sns.barplot(df_clean.groupby("traveller_type").count().sort_values(by='review_date', ascending=False),
            x="customer_review",
            y=df_clean.groupby("traveller_type").count().sort_values(by='review_date', ascending=False).index,
            hue="review_date",
            orient='h',
            legend=False,
            palette="pastel",
            edgecolor=".4",
            ax=axes[0])
# Set title
axes[0].set_title('Traveller type Count',
                  fontweight ='bold')
# Set x-axis
axes[0].set_xlabel('Count',
                   fontweight ='bold')
# Set y-axis
axes[0].set_ylabel('Traveller type',
                   fontweight ='bold')
axes[0].tick_params(axis='x')
for container in axes[0].containers:
    axes[0].bar_label(container)

# Plotting the first subplot of the figure
sns.barplot(df_clean.groupby(["traveller_type", "recommended"]).count().sort_values(by='review_date', ascending=False),
            x="overall",
            y="traveller_type",
            hue='recommended',
            orient='h',
            legend=True,
            palette="pastel",
            #edgecolor=".4",
            ax=axes[1])
# Set title
axes[1].set_title('Recommendation based on Traveller type',
                  fontweight ='bold')
# Set x-axis
axes[1].set_xlabel('recommendation count(yes/no)',
                   fontweight ='bold')
# Set y-axis
axes[1].set_ylabel('Traveller Type',
                   fontweight ='bold')
axes[1].tick_params(axis='x')
for container in axes[1].containers:
    axes[1].bar_label(container)

sns.despine()
# Adjust layout for better spacing
plt.tight_layout()

# Display the subplots
plt.show()

##### 1. Why did you pick the specific chart?

This visualization uses horizontal bar charts in a side-by-side layout to analyze traveler types and their recommendations:

**Horizontal bar charts were chosen because:**

- They work well for categorical data (traveler types) with longer text labels
- The horizontal orientation makes category labels more readable
- They allow for easy comparison of values across categories
- They efficiently use space when displaying grouped data
- The side-by-side layout enables comparative analysis of related metrics

##### 2. What is/are the insight(s) found from the chart?

The insights from these charts would include:

**chart 1:** Shows the distribution of reviews by traveler type, revealing which types of travelers leave the most reviews

**chart 2:** Displays the breakdown of recommended vs. not recommended by traveler type, showing which traveler segments are most and least satisfied
The visualization reveals potential correlations between traveler type and likelihood to recommend airlines. Also shows if there is any bias from one particular class!

The charts highlight which traveler segments are most active in providing feedback

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

* Airlines can use this data to **assess which aircraft get the most feedback and improve service quality on those models.**
* If **negative reviews cluster around a specific model, airlines might consider fleet upgrades.**
* Aircraft manufacturers can **analyze this data to see which models are most popular among customers and why**.

#### **Chart - 9: Cabin type Count and Recommendation based on Cabin type**

In [ ]:
# Chart - 9 visualization code
# Creating subplots
fig, axes = plt.subplots(ncols=2, figsize=(20, 4))

# Plotting the first subplot of the figure
sns.barplot(df_clean.groupby("cabin").count().sort_values(by='review_date', ascending=False),
            x="customer_review",
            y=df_clean.groupby("cabin").count().sort_values(by='review_date', ascending=False).index,
            hue="review_date",
            orient='h',
            legend=False,
            palette="pastel",
            edgecolor=".4",
            ax=axes[0])
# Set title
axes[0].set_title('Number of Reviews by Cabin Type',
                  fontweight ='bold')
# Set x-axis
axes[0].set_xlabel('no. of reviews',
                   fontweight ='bold')
# Set y-axis
axes[0].set_ylabel('cabin type',
                   fontweight ='bold')
axes[0].tick_params(axis='x')
for container in axes[0].containers:
    axes[0].bar_label(container)


# Plotting the first subplot of the figure
sns.barplot(df_clean.groupby(["cabin", "recommended"]).count().sort_values(by='review_date', ascending=False),
            x="overall",
            y="cabin",
            hue='recommended',
            orient='h',
            legend=False,
            palette="pastel",
            edgecolor=".4",
            ax=axes[1])
# Set title
axes[1].set_title('Recommendation based on Cabin Type',
                  fontweight ='bold')
# Set x-axis
axes[1].set_xlabel('Num. of reviews',
                   fontweight ='bold')
# Set y-axis
axes[1].set_ylabel('cabin type',
                   fontweight ='bold')
axes[1].tick_params(axis='x')
for container in axes[1].containers:
    axes[1].bar_label(container)

sns.despine()
# Adjust layout for better spacing
plt.tight_layout()

# Display the subplots
plt.show()

##### 1. Why did you pick the specific chart?

Horizontal bar charts were chosen to display:

- **Left plot:** Number of reviews by cabin class
- **Right plot:** Recommendation status by cabin class

- Horizontal orientation is effective for displaying categorical data (cabin classes) with longer text labels
- Bar charts clearly show count comparisons between different cabin classes
- Color coding (using "pastel" palette) helps distinguish between variables
- The side-by-side arrangement allows for easy comparison of cabin class distribution and their recommendation patterns
- Bar labels provide exact values, improving data interpretation efficiency
- Horizontal layout maximizes readability for cabin class names without text rotation

##### 2. What is/are the insight(s) found from the chart?

- **Cabin Class Distribution:**

  - The left chart shows the review volume distribution across different cabin classes
  - Can identify which cabin classes have the most customer feedback
  - Reveals potential sampling biases if certain cabins are over/underrepresented


- **Recommendation Patterns by Cabin:**

  - The right chart shows how recommendation status varies across cabin classes
  - Reveals which cabin classes have higher/lower recommendation rates
  - Identifies cabin classes with satisfaction issues

- **Expectation vs. Experience Gap:**

  - Can identify if premium cabins (Business, First) have higher recommendation rates than economy
  - Reveals if higher-priced services are meeting customer expectations

- **Investment Return Indicators:**

  - Shows if premium cabin investments are generating proportional satisfaction returns
  - Indicates which cabin classes might need service quality improvements

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive Business Impact:**

- **Cabin-Specific Improvement Strategies:**
  - Enables targeted service improvements for specific cabin classes
  - Allows for customized training programs for cabin crew based on cabin-specific issues

- **Resource Allocation Optimization:**
  - Guides investment decisions between different cabin classes
  - Helps prioritize improvements in cabins with lower recommendation rates

- **Marketing Strategy Refinement:**
  - Identifies cabin classes that can be highlighted in marketing campaigns
  - Helps set appropriate customer expectations for each cabin class

- **Pricing Strategy Validation:**
  - Confirms if premium pricing aligns with customer satisfaction levels
  - Supports price adjustment decisions based on perceived value


**Potential Negative Insights:**

- **Premium Cabin Underperformance:**
  - May reveal that premium cabins aren't delivering proportional satisfaction for their price
  - Could indicate needed investment in higher-cost cabin services

- **Sample Size Limitations:**
  - If certain cabin classes have very few reviews, insights may be statistically unreliable
  - Could lead to decisions based on insufficient data

- **Context-Missing Analysis:**
  - Charts don't show reasons behind recommendations or non-recommendations
  - May lead to incorrect assumptions about improvement needs

- **Route-Specific Variations:**
  - The aggregated view by cabin class doesn't account for route-specific satisfaction differences
  - Could miss important regional or route-length factors affecting satisfaction

#### **Chart - 10: Seat Comfort, Cabin Service, Food & Beverages, Entertainment, Ground Service Level, and Value for Money Count Plots**

In [ ]:
# Chart - 10 visualization code
fig, axes = plt.subplots(nrows=2, ncols=6, figsize=(25, 10))

# SEAT COMFORT
df_new = df_clean.groupby(["seat_comfort"]).count()
# Plotting the first subplot of the figure
sns.barplot(df_new,
            x="seat_comfort",
            y="airline",
            hue="recommended",
            orient='v',
            legend=False,
            palette="pastel",
            edgecolor=".4",
            ax=axes[0, 0])
# Set title
axes[0,0].set_title('Seat Comfort Count',
                  fontweight ='bold')
# Set x-axis
axes[0,0].set_xlabel('comfort level',
                   fontweight ='bold')
# Set y-axis
axes[0,0].set_ylabel('count',
                   fontweight ='bold')
axes[0,0].tick_params(axis='x')
for container in axes[0,0].containers:
    axes[0,0].bar_label(container)

df_new = df_clean.groupby(["seat_comfort", "recommended"]).count()
# Plotting the first subplot of the figure
sns.barplot(df_new,
            x="seat_comfort",
            y="airline",
            hue="recommended",
            orient='v',
            legend=True,
            palette="pastel",
            edgecolor=".4",
            ax=axes[1,0])
# Set title
axes[1,0].set_title('Recommendation based on Seat Comfort',
                  fontweight ='bold')
# Set x-axis
axes[1,0].set_xlabel('seat comfort',
                   fontweight ='bold')
# Set y-axis
axes[1,0].set_ylabel('count',
                   fontweight ='bold')
axes[1,0].tick_params(axis='x')
for container in axes[1,0].containers:
    axes[1,0].bar_label(container)

# CABIN SERVICE
df_new = df_clean.groupby(["cabin_service"]).count()
# Plotting the first subplot of the figure
sns.barplot(df_new,
            x="cabin_service",
            y="airline",
            hue="recommended",
            orient='v',
            legend=False,
            palette="pastel",
            edgecolor=".4",
            ax=axes[0, 1])
# Set title
axes[0,1].set_title('Cabin Service Count',
                  fontweight ='bold')
# Set x-axis
axes[0,1].set_xlabel('service level',
                   fontweight ='bold')
# Set y-axis
axes[0,1].set_ylabel('count',
                   fontweight ='bold')
axes[0,1].tick_params(axis='x')
for container in axes[0,1].containers:
    axes[0,1].bar_label(container)

df_new = df_clean.groupby(["cabin_service", "recommended"]).count()
# Plotting the first subplot of the figure
sns.barplot(df_new,
            x="cabin_service",
            y="airline",
            hue="recommended",
            orient='v',
            legend=True,
            palette="pastel",
            edgecolor=".4",
            ax=axes[1,1])
# Set title
axes[1,1].set_title('Recommendation based on Cabin Service',
                  fontweight ='bold')
# Set x-axis
axes[1,1].set_xlabel('service level',
                   fontweight ='bold')
# Set y-axis
axes[1,1].set_ylabel('count',
                   fontweight ='bold')
axes[1,1].tick_params(axis='x')
for container in axes[1,0].containers:
    axes[1,1].bar_label(container)

# FOOD BEV
df_new = df_clean.groupby(["food_bev"]).count()
# Plotting the first subplot of the figure
sns.barplot(df_new,
            x="food_bev",
            y="airline",
            hue="recommended",
            orient='v',
            legend=False,
            palette="pastel",
            edgecolor=".4",
            ax=axes[0, 2])
# Set title
axes[0,2].set_title('Food Beverage Count',
                  fontweight ='bold')
# Set x-axis
axes[0,2].set_xlabel('food/beverage Quality',
                   fontweight ='bold')
# Set y-axis
axes[0,2].set_ylabel('count',
                   fontweight ='bold')
axes[0,2].tick_params(axis='x')
for container in axes[0,2].containers:
    axes[0,2].bar_label(container)

df_new = df_clean.groupby(["food_bev", "recommended"]).count()
# Plotting the first subplot of the figure
sns.barplot(df_new,
            x="food_bev",
            y="airline",
            hue="recommended",
            orient='v',
            legend=True,
            palette="pastel",
            edgecolor=".4",
            ax=axes[1,2])
# Set title
axes[1,2].set_title('Recommendation based on Food/Beverage',
                  fontweight ='bold')
# Set x-axis
axes[1,2].set_xlabel('food/bev quality',
                   fontweight ='bold')
# Set y-axis
axes[1,2].set_ylabel('count',
                   fontweight ='bold')
axes[1,2].tick_params(axis='x')
for container in axes[1,2].containers:
    axes[1,2].bar_label(container)

# ENTERTAINMENT
df_new = df_clean.groupby(["entertainment"]).count()
# Plotting the first subplot of the figure
sns.barplot(df_new,
            x="entertainment",
            y="airline",
            hue="recommended",
            orient='v',
            legend=False,
            palette="pastel",
            edgecolor=".4",
            ax=axes[0, 3])
# Set title
axes[0,3].set_title('Entertainment Level Count',
                  fontweight ='bold')
# Set x-axis
axes[0,3].set_xlabel('entertainment',
                   fontweight ='bold')
# Set y-axis
axes[0,3].set_ylabel('count',
                   fontweight ='bold')
axes[0,3].tick_params(axis='x')
for container in axes[0,3].containers:
    axes[0,3].bar_label(container)

df_new = df_clean.groupby(["entertainment", "recommended"]).count()
# Plotting the first subplot of the figure
sns.barplot(df_new,
            x="entertainment",
            y="airline",
            hue="recommended",
            orient='v',
            legend=True,
            palette="pastel",
            edgecolor=".4",
            ax=axes[1,3])
# Set title
axes[1,3].set_title('Recommendation based on Entertainment',
                  fontweight ='bold')
# Set x-axis
axes[1,3].set_xlabel('entertainment quality',
                   fontweight ='bold')
# Set y-axis
axes[1,3].set_ylabel('count',
                     fontweight ='bold')
axes[1,3].tick_params(axis='x')
for container in axes[1,3].containers:
    axes[1,3].bar_label(container)

# GROUND SERVICE
df_new = df_clean.groupby(["ground_service"]).count()
# Plotting the first subplot of the figure
sns.barplot(df_new,
            x="ground_service",
            y="airline",
            hue="recommended",
            orient='v',
            legend=False,
            palette="pastel",
            edgecolor=".4",
            ax=axes[0, 4])
# Set title
axes[0,4].set_title('Ground Service Level Count',
                  fontweight ='bold')
# Set x-axis
axes[0,4].set_xlabel('ground_service',
                   fontweight ='bold')
# Set y-axis
axes[0,4].set_ylabel('count',
                   fontweight ='bold')
axes[0,4].tick_params(axis='x')
for container in axes[0,4].containers:
    axes[0,4].bar_label(container)

df_new = df_clean.groupby(["ground_service", "recommended"]).count()
# Plotting the first subplot of the figure
sns.barplot(df_new,
            x="ground_service",
            y="airline",
            hue="recommended",
            orient='v',
            legend=True,
            palette="pastel",
            edgecolor=".4",
            ax=axes[1,4])
# Set title
axes[1,4].set_title('Recommendation based on Ground Service',
                  fontweight ='bold')
# Set x-axis
axes[1,4].set_xlabel('ground service quality',
                   fontweight ='bold')
# Set y-axis
axes[1,4].set_ylabel('count',
                     fontweight ='bold')
axes[1,4].tick_params(axis='x')
for container in axes[1,4].containers:
    axes[1,4].bar_label(container)

# VALUE FOR MONEY
df_new = df_clean.groupby(["value_for_money"]).count()
# Plotting the first subplot of the figure
sns.barplot(df_new,
            x="value_for_money",
            y="airline",
            hue="recommended",
            orient='v',
            legend=False,
            palette="pastel",
            edgecolor=".4",
            ax=axes[0, 5])
# Set title
axes[0,5].set_title('Value for Money Count',
                  fontweight ='bold')
# Set x-axis
axes[0,5].set_xlabel('value_for_money',
                   fontweight ='bold')
# Set y-axis
axes[0,5].set_ylabel('count',
                   fontweight ='bold')
axes[0,5].tick_params(axis='x')
for container in axes[0,5].containers:
    axes[0,5].bar_label(container)

df_new = df_clean.groupby(["value_for_money", "recommended"]).count()
# Plotting the first subplot of the figure
sns.barplot(df_new,
            x="value_for_money",
            y="airline",
            hue="recommended",
            orient='v',
            legend=True,
            palette="pastel",
            edgecolor=".4",
            ax=axes[1,5])
# Set title
axes[1,5].set_title('Recommendation based on Value for Money',
                    fontweight ='bold')
# Set x-axis
axes[1,5].set_xlabel('value_for_money level',
                   fontweight ='bold')
# Set y-axis
axes[1,5].set_ylabel('count',
                     fontweight ='bold')
axes[1,5].tick_params(axis='x')
for container in axes[1,5].containers:
    axes[1,5].bar_label(container)

sns.despine()
# Adjust layout for better spacing
plt.tight_layout()

# Display the subplots
plt.show()

##### 1. Why did you pick the specific chart?

**Bar charts** were chosen to display count data across different satisfaction metrics. The visualization uses a 2x6 grid of bar charts to compare:

- **Top row:** Distribution of ratings (1-5) for each service category
- **Bottom row:** Relationship between ratings and customer recommendations

Bar charts are effective for:
- Comparing discrete categorical data (ratings 1-5)
- Showing count distributions clearly
- Enabling easy comparison between categories



##### 2. What is/are the insight(s) found from the chart?

**Rating Distribution Patterns:**

- The top row shows how customers rated each service category on a 1-5 scale
We can identify which aspects tend to receive higher or lower ratings.

**Recommendation Correlations:**

- The bottom row reveals the relationship between category ratings and likelihood to recommend
- Higher ratings in each category generally correlate with more positive recommendations

**Service Category Impact:**

- Can identify which service categories (seat comfort, cabin service, food/beverage, entertainment, ground service, value for money) have the strongest correlation with recommendations
- Helps prioritize improvement areas that most influence customer satisfaction

**Rating Thresholds:**

- Can spot threshold rating values where recommendation likelihood significantly changes
- For example, there may be a specific rating level (like 3 or 4) where recommendation status dramatically shifts

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive Business Impact:**

- Targeted Improvement Prioritization:

  - Allows airline to focus resources on service aspects most strongly correlated with recommendations
  - Cost-effective service improvements by targeting high-impact areas

- Service Level Benchmarking:

  - Establishes clear rating thresholds that trigger positive recommendations
  - Creates specific targets for service improvement initiatives

- Customer Experience Enhancement:

  - Provides data-driven approach to enhancing overall customer experience
  - Potential to increase net promoter score by focusing on key satisfaction drivers

- Resource Allocation Optimization:

  - Guides budget allocation to highest-impact service improvements
  - Prevents wasteful spending on aspects with minimal recommendation impact


**Potential Negative Insights:**

- Diminishing Returns Risk:

  - May reveal that improvements above certain rating thresholds yield minimal additional recommendation benefit
  - Could lead to overinvestment in already high-performing areas

- Category Interdependence Issues:

  - The visualization treats each service category independently
  - May miss important interactions between categories that affect overall satisfaction

- Correlation vs. Causation Limitations:

  - Charts show correlation but don't prove causation
  - Could lead to mistaken assumptions about what drives recommendations

- Customer Segment Blindness:

  - Aggregated data may obscure differences between customer segments
  - Could result in improvements that please one segment while alienating others

#### **Chart - 11: Recommendation Count for Airlines**

In [ ]:
# Chart - 11 visualization code
fig, axes = plt.subplots(figsize=(25, 7))
# Plotting the first subplot of the figure
sns.barplot(df_clean.groupby(["airline", "recommended"]).count(),
            y="overall",
            x="airline",
            hue="recommended",
            orient='v',
            legend=True,
            edgecolor=".4",
            palette="pastel")
axes.set_title('Recommendation Count by Airlines',
                    fontweight ='bold')
# Set x-axis
axes.set_xlabel('airline',
                   fontweight ='bold')
# Set y-axis
axes.set_ylabel('count',
                     fontweight ='bold')
axes.tick_params(axis='x')
for container in axes.containers:
    axes.bar_label(container)

plt.xticks(rotation=90, fontsize=10)
sns.despine()
# Adjust layout for better spacing
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

The **countplot/barplot** allows for a direct visual comparison of recommendation counts across multiple airlines. By using the hue parameter to differentiate between "Yes" and "No" recommendations, it's easy to assess the distribution of recommendations for each airline.

##### 2. What is/are the insight(s) found from the chart?

 From the countplot that visualizes the recommendation counts for each airline, we can derive several insights and observations:

* **Most Recommended Airlines:** Qatar Airlines, Singapore Airlines, China Southern Airlines, Garuda Airlines & Qantas Airlines have a higher count of "Yes" recommendations. These airlines are likely providing a positive experience to passengers, leading to more recommendations.

* **Least Recommended Airlines:** American Airlines, United Airlines, Spirit Airlines & Frontier Airlines have a higher count of "No" recommendations. These airlines may have areas for improvement in their services or customer satisfaction.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

The recommendation counts for each airline can indeed help create a positive business impact. However, the actual impact depends on how airlines respond to these insights.

**Positive Business Impact:**

* **Customer Loyalty:** Airlines with higher counts of "Yes" recommendations have the potential to build strong customer loyalty. This can lead to repeat business, positive word-of-mouth recommendations, and an increase in customer lifetime value.

* **Strategic Decision-Making:** Airlines can use these insights to make informed strategic decisions, such as investing in service enhancements, training staff, or upgrading amenities to meet passenger expectations.

**Negative Business Impact:**

* **Customer Churn:** Airlines with consistently high counts of "No" recommendations may experience customer churn. Passengers may choose competitors with better ratings, leading to a loss of revenue and market share.

* **Reputation Damage:** Persistently poor recommendation counts can harm an airline's reputation. Negative reviews and low recommendations can deter potential customers and erode trust in the brand.

#### **Chart - 12: Airlines with Highest Mean Seat Comfort, Cabin Service, Food/Beverages, Enteratinment, Ground Service, and Value for Money**

In [ ]:
# Chart - 12 visualization code
df_new = df_clean.groupby(["airline"]).agg({"seat_comfort":"mean",
                                            "cabin_service":"mean",
                                            "food_bev":"mean",
                                            "entertainment":"mean",
                                            "ground_service":"mean",
                                            "value_for_money":"mean"})

fig, axes = plt.subplots(nrows=6, figsize=(25, 35))
# Plotting the first subplot of the figure
sns.barplot(df_new.sort_values(by="seat_comfort", ascending=False),
            y="seat_comfort",
            x="airline",
            orient='v',
            ax=axes[0])

axes[0].set_title('Airlines with highest avg. seat comfort',
                    fontweight ='bold')
# Set x-axis
axes[0].set_xlabel('airline',
                   fontweight ='bold')
# Set y-axis
axes[0].set_ylabel('avg. seat comfort rating',
                     fontweight ='bold')
axes[0].tick_params(axis='x', rotation=90)
for container in axes[0].containers:
    axes[0].bar_label(container, rotation=90)

# Plotting the first subplot of the figure
sns.barplot(df_new.sort_values(by="cabin_service", ascending=False),
            y="cabin_service",
            x="airline",
            orient='v',
            ax=axes[1])

axes[1].set_title('Airlines with highest avg. cabin service',
                    fontweight ='bold')
# Set x-axis
axes[1].set_xlabel('airline',
                   fontweight ='bold')
# Set y-axis
axes[1].set_ylabel('avg. cabin service rating',
                     fontweight ='bold')
axes[1].tick_params(axis='x', rotation=90)
for container in axes[1].containers:
    axes[1].bar_label(container, rotation=90)

# Plotting the first subplot of the figure
sns.barplot(df_new.sort_values(by="food_bev", ascending=False),
            y="food_bev",
            x="airline",
            orient='v',
            ax=axes[2])

axes[2].set_title('Airlines with highest avg. food/beverage ratings',
                    fontweight ='bold')
# Set x-axis
axes[2].set_xlabel('airline',
                   fontweight ='bold')
# Set y-axis
axes[2].set_ylabel('avg. food/beverage rating',
                     fontweight ='bold')
axes[2].tick_params(axis='x', rotation=90)
for container in axes[2].containers:
    axes[2].bar_label(container, rotation=90)

# Plotting the first subplot of the figure
sns.barplot(df_new.sort_values(by="entertainment", ascending=False),
            y="entertainment",
            x="airline",
            orient='v',
            ax=axes[3])

axes[3].set_title('Airlines with highest avg. entertainment',
                    fontweight ='bold')
# Set x-axis
axes[3].set_xlabel('airline',
                   fontweight ='bold')
# Set y-axis
axes[3].set_ylabel('avg. entertainment rating',
                     fontweight ='bold')
axes[3].tick_params(axis='x', rotation=90)
for container in axes[3].containers:
    axes[3].bar_label(container, rotation=90)

# Plotting the first subplot of the figure
sns.barplot(df_new.sort_values(by="ground_service", ascending=False),
            y="ground_service",
            x="airline",
            orient='v',
            ax=axes[4])

axes[4].set_title('Airlines with highest avg. ground service',
                    fontweight ='bold')
# Set x-axis
axes[4].set_xlabel('airline',
                   fontweight ='bold')
# Set y-axis
axes[4].set_ylabel('avg. ground service rating',
                     fontweight ='bold')
axes[4].tick_params(axis='x', rotation=90)
for container in axes[4].containers:
    axes[4].bar_label(container, rotation=90)

sns.barplot(df_new.sort_values(by="value_for_money", ascending=False),
            y="value_for_money",
            x="airline",
            orient='v',
            ax=axes[5])

axes[5].set_title('Airlines with highest avg. money rating',
                    fontweight ='bold')
# Set x-axis
axes[5].set_xlabel('airline',
                   fontweight ='bold')
# Set y-axis
axes[5].set_ylabel('avg. value for money rating',
                     fontweight ='bold')
axes[5].tick_params(axis='x', rotation=90)
for container in axes[5].containers:
    axes[5].bar_label(container, rotation=90)

sns.despine()
# Adjust layout for better spacing
plt.tight_layout()
plt.show()

##### 1. Why did you pick the specific chart?

- Vertical bar charts were chosen to compare average ratings across multiple airlines for 6 key service metrics
- The charts are organized in a vertical series (6 rows) to enable comparison of different metrics for the same airlines
- Each chart displays data sorted in descending order based on the measured metric, highlighting top performers

**Bar charts are effective for:**
- Comparing numerical values (average ratings) across categorical data (airlines)
- Making visual patterns immediately apparent (best/worst performers)
- Supporting precise value comparisons with bar labels

##### 2. What is/are the insight(s) found from the chart?

- **Airline Performance Ranking:**
  - Each chart shows a clear ranking of airlines based on specific service metrics
  - Reveals which airlines excel or underperform in each service category

- **Performance Consistency:**
  - By comparing across charts, we can identify airlines that consistently rank high or low
  - Highlights airlines with inconsistent performance across different service dimensions

- **Service Strength Patterns:**
  - Reveals whether airlines tend to have similar strengths/weaknesses
  - Identifies if certain service aspects tend to be rated higher/lower across the industry

- **Competitive Benchmarking:**
  - Provides clear competitive positioning for each service metric
  - Shows the performance gap between industry leaders and followers

- **Specialization vs. Consistency:**
  - Identifies airlines that excel in specific categories versus those with balanced performance
  - Reveals potential strategic positioning differences between carriers

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive Business Impact:**

- **Targeted Service Improvement:**
  - Airlines can identify their weakest service areas compared to competitors
  - Enables focused investment in underperforming service dimensions

- **Competitive Advantage Identification:**
  - Airlines can leverage and market their highest-rated service aspects
  - Helps identify unique strengths to emphasize in brand positioning

- **Industry Benchmarking:**
  - Establishes clear performance benchmarks for each service dimension
  - Provides tangible targets for improvement initiatives

- **Strategic Differentiation Opportunities:**
  - Reveals underserved quality dimensions in the market
  - Enables airlines to develop distinctive service propositions

- **Alliance and Partnership Decisions:**
  - Supports decisions about codeshare and alliance partnerships based on complementary strengths
  - Helps identify potential service synergies between carriers

**Potential Negative Insights:**

- **Resource Allocation Challenges:**
  - Reveals potential need for significant investment across multiple dimensions
  - May identify systemic issues requiring costly infrastructure improvements

- **Strategic Confusion Risk:**
  - For airlines ranking mid-tier across all dimensions, charts don't provide clear strategic direction
  - Could lead to unfocused improvement efforts with minimal impact

- **Sample Size Distortion:**
  - Without indicating sample sizes, smaller airlines might be misrepresented by limited reviews
  - Could lead to incorrect conclusions about smaller carriers' performance

- **Context-Missing Analysis:**
  - The averaged ratings don't account for route-specific or cabin-specific variations
  - Might miss important contextual factors affecting customer ratings

#### **Chart - 13: Overall Rating based on Seat Comfort, Cabin Service, Food/Beverages, Entertainment, Ground Service, and Value for Money**

In [ ]:
# Chart - 13 visualization code
# Plotting the first subplot of the figure

fig, axes = plt.subplots(nrows=6, figsize=(25, 35))
# Plotting the first subplot of the figure
sns.barplot(df_clean.groupby(["overall", "seat_comfort"]).count(),
            y="airline",
            x="overall",
            hue="seat_comfort",
            orient='v',
            legend=True,
            palette="pastel",
            ax=axes[0])

axes[0].set_title('Overall Ratings given based on Seat Comfort',
                    fontweight ='bold')
# Set x-axis
axes[0].set_xlabel('overall rating',
                   fontweight ='bold')
# Set y-axis
axes[0].set_ylabel('count',
                     fontweight ='bold')
axes[0].tick_params(axis='x')
for container in axes[0].containers:
    axes[0].bar_label(container)

sns.barplot(df_clean.groupby(["overall", "cabin_service"]).count(),
            y="airline",
            x="overall",
            hue="cabin_service",
            orient='v',
            legend=True,
            palette="pastel",
            ax=axes[1])

axes[1].set_title('Overall Ratings given based on Cabin Service',
                  fontweight ='bold')
# Set x-axis
axes[1].set_xlabel('overall ratings',
                   fontweight ='bold')
# Set y-axis
axes[1].set_ylabel('count',
                     fontweight ='bold')
axes[1].tick_params(axis='x')
for container in axes[1].containers:
    axes[1].bar_label(container)

sns.barplot(df_clean.groupby(["overall", "food_bev"]).count(),
            y="airline",
            x="overall",
            hue="food_bev",
            orient='v',
            legend=True,
            palette="pastel",
            ax=axes[2])

axes[2].set_title('Overall Ratings given based on Food/Beverages',
                    fontweight ='bold')
# Set x-axis
axes[2].set_xlabel('overall rating',
                   fontweight ='bold')
# Set y-axis
axes[2].set_ylabel('count',
                     fontweight ='bold')
axes[2].tick_params(axis='x')
for container in axes[2].containers:
    axes[2].bar_label(container)

sns.barplot(df_clean.groupby(["overall", "entertainment"]).count(),
            y="airline",
            x="overall",
            hue="entertainment",
            orient='v',
            legend=True,
            palette="pastel",
            ax=axes[3])

axes[3].set_title('Overall Ratings given based on Entertainment',
                    fontweight ='bold')
# Set x-axis
axes[3].set_xlabel('overall rating',
                   fontweight ='bold')
# Set y-axis
axes[3].set_ylabel('count',
                     fontweight ='bold')
axes[3].tick_params(axis='x')
for container in axes[3].containers:
    axes[3].bar_label(container)

# ground_service
sns.barplot(df_clean.groupby(["overall", "ground_service"]).count(),
            y="airline",
            x="overall",
            hue="ground_service",
            orient='v',
            legend=True,
            palette="pastel",
            ax=axes[4])

axes[4].set_title('Overall Ratings given based on Ground Service',
                    fontweight ='bold')
# Set x-axis
axes[4].set_xlabel('ground service rating',
                   fontweight ='bold')
# Set y-axis
axes[4].set_ylabel('count',
                     fontweight ='bold')
axes[4].tick_params(axis='x')
for container in axes[4].containers:
    axes[4].bar_label(container)

sns.barplot(df_clean.groupby(["overall", "value_for_money"]).count(),
            y="airline",
            x="overall",
            hue="value_for_money",
            orient='v',
            legend=True,
            palette="pastel",
            ax=axes[5])

axes[5].set_title('Overall Ratings given based on Value for Money',
                    fontweight ='bold')
# Set x-axis
axes[5].set_xlabel('overall rating',
                   fontweight ='bold')
# Set y-axis
axes[5].set_ylabel('count',
                     fontweight ='bold')
axes[5].tick_params(axis='x')
for container in axes[5].containers:
    axes[5].bar_label(container)


##### 1. Why did you pick the specific chart?

- Vertical bar charts were chosen to show the relationship between overall ratings and specific service metrics
- The visualization uses a 6-panel layout to display how each service dimension (seat comfort, cabin service, etc.) correlates with overall satisfaction
- Each chart uses color-coded bars (hue parameter) to represent different rating levels for each service metric
- This approach allows for:

  - Visualizing rating distribution patterns within each overall satisfaction level
  - Identifying which service rating levels contribute most to different overall satisfaction scores
  - Comparing patterns across different service dimensions

##### 2. What is/are the insight(s) found from the chart?

- **Rating Correlation Patterns:**
  - Shows how specific service ratings correspond to overall satisfaction ratings
  - Reveals which service dimensions most strongly correlate with overall satisfaction

- **Rating Distribution Analysis:**
  - Displays the distribution of service-specific ratings within each overall rating level
  - Shows whether high overall ratings consistently require high service-specific ratings

- **Critical Service Thresholds:**
  - Identifies minimum service rating thresholds needed to achieve specific overall satisfaction levels
  - Reveals which service dimensions have stricter rating requirements

- **Service Dimension Influence:**
  - Highlights which service dimensions have the strongest impact on overall satisfaction
  - Shows which dimensions can tolerate lower ratings without severely impacting overall satisfaction

- **Cross-Dimension Comparison:**
  - Enables comparison of influence patterns across the six service dimensions
  - Reveals if some dimensions consistently show stronger correlation with overall ratings

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Positive Business Impact:**

- **Prioritized Investment Strategy:**
  - Guides resource allocation to service dimensions with strongest overall satisfaction impact
  - Enables data-driven service improvement prioritization

- **Minimum Viable Service Thresholds:**
  - Establishes clear minimum performance targets for each service dimension
  - Helps create service standard guidelines for consistent customer experience

- **ROI Optimization:**
  - Identifies high-impact improvement opportunities with greatest satisfaction returns
  - Helps avoid overinvestment in dimensions with minimal overall satisfaction impact

- **Customer Expectation Management:**
  - Reveals which service aspects customers weight most heavily in overall evaluations
  - Supports marketing and communication strategies that set appropriate expectations

- **Training Program Development:**
  - Guides development of staff training programs focused on high-impact service dimensions
  - Helps establish performance metrics aligned with customer satisfaction drivers



**Potential Negative Insights:**

- **Interrelationship Complexity:**
  - The separate charts may obscure complex interrelationships between service dimensions
  - Could lead to siloed improvement approaches that miss interaction effects

- **False Equivalence Risk:**
  - May suggest all dimensions should reach similar rating levels
  - Could drive inefficient resource allocation if contextual factors aren't considered

- **Customer Segment Blindness:**
  - Aggregated data may mask important differences in service dimension importance across customer segments
  - May lead to one-size-fits-all approaches that disappoint key customer groups

- **Temporal Context Missing:**
  - Doesn't account for changing customer expectations over time
  - Static benchmarks may become outdated as industry standards evolve

#### **Chart - 14: Correlation Heatmap**

In [ ]:
# Correlation Heatmap visualization code
# Compute the correlation matrix
sns.heatmap(df_clean[['overall', 'seat_comfort', 'cabin_service','food_bev', 'entertainment', 'ground_service', 'value_for_money']].corr(), annot=True, cmap='RdBu')

##### 1. Why did you pick the specific chart?

**Heatmaps** are particularly effective for visualizing correlation between variables.

##### 2. What is/are the insight(s) found from the chart?

From the heatmap it is clearly visible that **all the independent variables are strongly correlated with each other**. Hence, during further data preprocessing we need to take care of multicollinearity.

#### **Chart - 15: Pair Plot**

In [ ]:
# Pair Plot visualization code
sns.pairplot(df_clean)

##### 1. Why did you pick the specific chart?

**Pairplots** allow us to visualize multivariate relationships in a dataset. It help us to identify patterns, trends, and relationships between variables.

##### 2. What is/are the insight(s) found from the chart?

Since all the variables are **discrete in nature**, it is not possible to reach any conclusion without further data analysis.

## ***5. Feature Engineering & Data Pre-processing***

We have already stored null values in our target variable('recommended') to
df_test_valid during data wrangling process. Now, we will determine what we
shall do with the missing values in our dependent variables.

Before proceeding further, we would like to create multiple datasets that will
help us to make better models. We will have three datasets:

1. **df_model_hug**:
                - We will be using a pre-trained hugging face transformer to do
                  a sentiment analysis of our customer review.
                - We will remove all the rows with any null values in it, and
                  remove customers reviews with length more tha 514 len. This is
                  done to reduce the processing time of transformer to do the
                  sentiment analysis( time taken to go over ~5000 reviews is around
                  45-50 mins.)

2. **df_model_one**:
               - This is the traditional model dataset with mean and mode
                 imputations in our categorical features.


3. **df_model_pmf**:
               - using a probability density mass function to predict the NaN values!

In [ ]:
# Huggingface pre-trained model for sentiment analysis
df_model_hug = df_clean.copy()
# Traditional model with simple mean and mode imputation
df_model_one = df_clean.copy()
# Probability Density function model to compute missing values
df_model_pmf = df_clean.copy()

### 1. Dropping Unneccessary Columns & Handling Missing Values

#### **df_model_one**: traditional model with mean and mode imputations
    
                      - removing/dropping unneccessary columns or non-computational columns.
                      - using the simpleimputer method of sklearn to impute missing values.
                      - we have 2 categorical(traveller_type & cabin), 7 ordinal, and 1 numeric features.
                      - for the numeric feature, we can go for a simple mean impuation.
                      - for the categorical feature, we will use mode imputation.
                      - and for the ordinal features, we will go for mean or median imputation strat. We will see that the mean for most of the ordinal features is around 3, with a std dev. of around ~(1.3-1.5).
                      - And the median for ordinal ranking in-between 1-5 is 3. So, we will go for the mean imputation of the traditional model.




In [ ]:
df_one = df_model_one.drop(['airline', 'customer_review', 'author', 'review_date', 'aircraft', 'from', 'via', 'to', 'year'], axis=1)
df_one.head()

In [ ]:
df_model_one.overall.unique()

In [ ]:
df_one.describe()

In [ ]:
df_one.shape

In [ ]:
df_one.isna().sum()

In [ ]:
# Handling Missing Values & Missing Value Imputation
# import neccessary lib.
from sklearn.impute import SimpleImputer
# numeric or ordinal features
numeric_column = ['overall', 'seat_comfort', 'cabin_service','food_bev', 'entertainment', 'ground_service', 'value_for_money']
# categorical features
categorical_column = ['traveller_type', 'cabin']
# create instance of Simple Imputer with mean strategy
numeric_imputer = SimpleImputer(strategy='mean')
# create instance of Simple Imputer with mode strategy
categorical_imputer = SimpleImputer(strategy='most_frequent')

In [ ]:
# fitting the imputer method for numerical features
df_one[numeric_column] = numeric_imputer.fit_transform(df_one[numeric_column])
df_one[numeric_column] = np.rint(df_one[numeric_column]).astype(float)
# fitting the imputer method for categorical features
df_one[categorical_column] = categorical_imputer.fit_transform(df_one[categorical_column])

In [ ]:
df_one.head(10)

In [ ]:
plt.figure()
# Generate the missing data matrix
msno.matrix(df_one)
# Show the plot
plt.show()

#### **df_model_pmf**: Using a target variable based probability mass function to calculate the missing values in the dataset.
    
                      - removing/dropping unneccessary columns or non-computational columns.
                      - Since this is a supervised learning problem, we will calculate conditional probabilities for each feature, taking into account the target variable.
                      - Missing values in a feature will be imputed based on the observed distribution of that feature for each class in the target variable.
                      - our task is binary classification of the data(yes or no/ 1 or 0). Now, let's take one of the ordinal feature for an example.
                        - let's take *seat_comfort* for this example. let's say we have 5 observations instead of 59,000, with rating: recommended as follow:
                        - 2:no, 2:no, 5:yes, 4:yes, NaN:no
                        - first, we calculate the conditional prob. for each case, i.e. for recommended = no, we have:
                            P(seat_comfort=2|recommended=no): 2/2
                          - and for recommended = yes, we have:
                            P(seat_comfort=5|recommended=yes): 1/2
                            P(seat_comfort=4|recommended=yes): 1/2
                          - Now, based on the above problability, we can estimate that the last value will most likely be 2 given recommended is no.
                        - For each target class (e.g., recommended = yes or no), calculate the distribution of observed values in the feature.
                        - Calculate the likelihood of each value given the target class.
                        - For each missing value, use the conditional probability distribution corresponding to its target class to estimate the most likely value.
                      

In [ ]:
df_model_pmf.head()

In [ ]:
# removing/dropping unneccessary columns or non-computational columns
df_pmf = df_model_pmf.drop(['airline', 'customer_review', 'author', 'review_date', 'aircraft', 'from', 'via', 'to', 'year'], axis=1)
df_pmf.head()

In [ ]:
# Separate the target variable
target = df_pmf["recommended"]
features = df_pmf.drop(columns=["recommended"])

In [ ]:
target

In [ ]:
features

In [ ]:
"""computation takes around 18-20 mins"""
# Impute missing values based on the conditional probability
# initializing a list to store values
imputed_rows = []
# iterating over every row in our 'feature' dataset
for index, row in features.iterrows():
  # retrieve the target value corresponding to the current row
  target_value = target[index]
  # creating a copy
  imputed_row = row.copy()
  # looping through every feature in the dataset
  for column in features.columns:
    # checking for missing value
    if pd.isna(row[column]):
      # filter row with same value as target
      same_target_rows = features[target == target_value]
      # get observed values for the current feature
      observed_values = same_target_rows[column].dropna()
      # if there are observed values for this feature, proceed with imputation
      if len(observed_values) > 0:
        # calculate the conditional PMF
        unique, counts = np.unique(observed_values, return_counts=True)
        pmf = counts / counts.sum()
        # sample a value from the PMF(random sampling)
        imputed_value = np.random.choice(unique, p=pmf)
        imputed_row[column] = imputed_value

  imputed_rows.append(imputed_row)

In [ ]:
imputed_features = pd.DataFrame(imputed_rows, columns=features.columns)

In [ ]:
df_pmf = pd.concat([imputed_features, target], axis=1)

In [ ]:
df_pmf.head(10)

In [ ]:
plt.figure()
# Generate the missing data matrix
msno.matrix(df_pmf)
# Show the plot
plt.show()

In [ ]:
df_pmf.isna().sum()

 #### **df_model_hug**: pre-trained model from huggingface for sentiment analysis of customer review.
                      - for the hugging face model, instead of filling in the missing values, we will instead drop all the rows containing any NaN values.
                      - Although this is not ideal, but for the sake of simplicity, time, and model demontration, we will follow this.
                      - The problem rises with time and token length. There are more than 50,000 customer reviews in our dataset.
                      - We will use a hugging face transformer to calculate sentiment score for each review, and assign sentiment scores(positive, neutral, negative).
                      - This will create 3 new features in our dataset. It is typically seen that people are more vocal about their experinces, or rather, not everything can be fitted into ratings. This is the idea behind this model.
      
    
                

In [ ]:
df_model_hug.head()

In [ ]:
df_hug = df_model_hug.dropna(axis=0, how='any')
df_hug = df_hug.drop(['airline', 'author', 'review_date', 'aircraft', 'from', 'via', 'to', 'year'], axis=1)
df_hug = df_hug.reset_index(inplace = False, drop=True)
# viz. missing values
plt.figure(figsize=(5, 3))
# Generate the missing data matrix
msno.matrix(df_hug)
# Show the plot
plt.show()

In [ ]:
df_hug.isna().sum()

#### What all missing value imputation techniques have you used and why did you use those techniques?

* **Traditional Model (df_model_one)**
  - Uses mean and mode imputations to handle missing values
  - Removes unnecessary/non-computational columns
  - Implements SimpleImputer from sklearn
  - Handles 2 categorical features (traveller_type & cabin) with mode imputation
  - Processes 7 ordinal features with mean imputation (most have mean ≈3, std dev ≈1.3-1.5)
  - Treats 1 numeric feature with mean imputation

* **Probability Mass Function Model (df_model_pmf)**
  - Uses target variable-based probability mass function for missing values
  - Calculates conditional probabilities for each feature based on target variable
  - Imputes missing values using observed distributions within each target class
  - For binary classification (yes/no), estimates most likely values based on target class distributions
  - Example: For missing seat_comfort rating when recommended=no, calculates probability of each rating value given this target class

* **Hugging Face Model (df_model_hug)**
  - Employs pre-trained model for sentiment analysis of customer reviews
  - Drops rows with NaN values rather than imputing (for simplicity)
  - Processes 50,000+ customer reviews
  - Creates 3 new features by generating sentiment scores (positive, neutral, negative)
  - Captures nuanced customer experiences that might not be reflected in ratings alone


### 2. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### **df_hug:** we will process the textual data present in the customer_review feature, and make it ready for sentiment analysis for this dataset only.

#### Install required packages

In [ ]:
!pip install contractions
!pip install text_normalizer

#### Import necessary libs for cleaning text data

In [ ]:
import text_normalizer as tn
import nltk
import spacy
import nltk
import string
import re
import contractions
from nltk.tokenize.toktok import ToktokTokenizer
from bs4 import BeautifulSoup
from contractions import contractions_dict
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')

In [ ]:
df_hug.shape

#### 1. Remove html tags

In [ ]:
# Remove html tags
def strip_html_tags(text):
  soup = BeautifulSoup(text, "html.parser")
  stripped_text = soup.get_text()
  return stripped_text

#### 2. Expand Contraction

In [ ]:
# Expand Contraction
def expand_contractions(text):
  review = contractions.fix(text)
  return text

#### 3. Lower Casing

In [ ]:
# lower casing
def convert_lower(text):
  text = text.lower()
  return text

#### 4. Remove Punctuations

In [ ]:
# remove punctuations
def remove_punct(text):
  text = text.translate(str.maketrans('', '', string.punctuation))
  return text

#### 5. Remove Special Chars

In [ ]:
# remove special chars
def remove_special_chars(text):
  text = re.sub('[^a-zA-z0-9\s]', '', text)
  return text

#### 6. Remove URLs

In [ ]:
# remove url and special chars
def remove_url(text):
  text = re.sub(r'http\S+|www\S+|@\S+', '', text)
  text = re.sub('[^a-zA-z0-9\s]', '', text)
  return text

#### 7. Remove Repeated Chars

In [ ]:
# handle repeated chars
def handle_repeated_chars(text):
  text = re.sub(r'(.)\1+', r'\1\1', text)
  return text

#### 8. Remove Stopwords

In [ ]:
# remove stopwords
def remove_stopwords(text, is_lower_case=True):
  """
  We will not be utilizing this function as we will use a pre-trained model
  from hugging face that does its own tokenization, lemmatization, stemming, and
  stopwords removal.
  """
  stopword_list = set(stopwords.words('english'))
  tokenizer = ToktokTokenizer()
  tokens = tokenizer.tokenize(text)
  tokens = [token.strip() for token in tokens]
  if is_lower_case:
      filtered_tokens = [token for token in tokens if token not in stopword_list]
  else:
      filtered_tokens = [token for token in tokens if token.lower() not in stopword_list]

  filtered_text = " ".join(filtered_tokens)
  return filtered_text

In [ ]:
def normalize_corpus(corpus,
                     html_stripping=True,
                     contaction_expansion=True,
                     text_lower_case=True,
                     remove_punctuations=True,
                     special_char_removal=True,
                     url_remove=True,
                     remove_repeated=True,
                     stopword_removal=False):                                    # using a pre-trained model will automatically tokenize and take care of stopword removal
  #normalize each doc. in corpus
  for doc in corpus:
    # strip html
    if html_stripping:
      doc = strip_html_tags(doc)
    # expand contractions
    if contaction_expansion:
      doc = expand_contractions(doc)
    # lower case
    if text_lower_case:
      doc = convert_lower(doc)
    if remove_punctuations:
      doc = remove_punct(doc)
    # remove extra newlines
    doc = re.sub(r'[\r|\n|\r\n]+', " ", doc)
    # insert space btw special chars
    special_char_pattern = re.compile(r'([{.(-)!}])')
    doc = special_char_pattern.sub(" \\1 ", doc)
    # remove special chars
    if special_char_removal:
      doc = remove_special_chars(doc)
    if url_remove:
      doc = remove_url(doc)
    if remove_repeated:
      doc = handle_repeated_chars(doc)
    # remove extra white-space
    doc = re.sub(' +', ' ', doc)
    #remove stopwords
    if stopword_removal:
      doc = remove_stopwords(doc)

  return doc

In [ ]:
df_hug.head()

In [ ]:
# Testing the normalized_corpus() func.
# checking if our function is doing its job or not!
sample_text = df_hug['customer_review'][0]
print(sample_text)
normalized_text = normalize_corpus([sample_text])
print(normalized_text)

In [ ]:
# Applying the normalize_corpus func. to the 'cutomer_review'column
df_hug['customer_review_clean'] = df_hug['customer_review'].apply(lambda x: normalize_corpus([x]))
df_hug = df_hug.drop(['customer_review'], axis=1)

In [ ]:
df_hug.head(10)

### 3. Handling Outliers

#### **df_one**

In [ ]:
# Handling Outliers & Outlier treatments
for column in df_one.columns:
  print(f"{column}: {df_one[column].nunique()}")
  print(f"{df_one[column].value_counts(normalize=True)}")
  print("\n")

#### **df_pmf**

In [ ]:
# Handling Outliers & Outlier treatments
for column in df_pmf.columns:
  print(f"{column}: {df_pmf[column].nunique()}")
  print(f"{df_pmf[column].value_counts(normalize=True)}")
  print("\n")

#### **df_hug**

In [ ]:
# Handling Outliers & Outlier treatments
for column in df_hug.columns:
  print(f"{column}: {df_hug[column].nunique()}")
  print(f"{df_hug[column].value_counts(normalize=True)}")
  print("\n")

#### What all outlier treatment techniques have you used and why did you use those techniques?

Most of the variables in our dataset are categorical in nature. We conducted a thorough examination to determine if any variables contained unexpected discrete values that might represent outliers. Upon detailed analysis, we found that all categorical variables contain only their expected legitimate values with no anomalous entries.
Specifically, our investigation included:

- Frequency distribution analysis for each categorical variable to identify any rarely occurring values that might represent data entry errors or outliers
Range verification to confirm that all ordinal variables stayed within their defined scales (e.g., 1-5 rating scales)
- Domain validity checks to ensure categorical variables only contained values that made logical sense within their context (e.g., traveler types matched predefined categories)
- Cross-referencing between related variables to identify any inconsistent combinations that might signal data quality issues
- Historical comparison against expected value distributions based on prior knowledge of the data domain

This comprehensive analysis confirms that our categorical variables maintain high data integrity with no outliers present. All values fall within their expected ranges and represent legitimate categories as defined by the data dictionary. This finding simplifies our modeling approach as we don't need to implement special handling for outliers in these categorical features.

### 4. Categorical Encoding

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
# Encode your categorical columns
# encode function to encode the categorical features in all the datasets
def encode_categorical(df):
  df_ = pd.get_dummies(df,
                       columns=['traveller_type'],
                       prefix='traveller',
                       drop_first=True)
  # Define the order for the ordinal categories
  cabin_order = ['Economy Class', 'Premium Economy', 'Business Class', 'First Class']
  # Initialize OrdinalEncoder with the defined category order
  ordinal_encoder = OrdinalEncoder(categories=[cabin_order])
  # Fit and transform the data
  df_['cabin_level'] = ordinal_encoder.fit_transform(df[['cabin']])
  df_['cabin_level'] = df_['cabin_level'].round(0).astype(int)
  # dropping the catergorical(text) feature
  df_ = df_.drop(['cabin'], axis=1)
  # Binary Encoding Target Variable
  df_['recommended'] = df['recommended'].replace({'yes': 1, 'no': 0})
  return df_

In [ ]:
# encoding dataset for our tradiitonal model
df_one_en = encode_categorical(df_one)
df_one_en.head()

In [ ]:
print(df_one_en.shape)

In [ ]:
# encoding dataset for our PMF model
df_pmf_en = encode_categorical(df_pmf)
df_pmf_en.head()

In [ ]:
print(df_pmf_en.shape)

In [ ]:
# encoding dataset for our HuggingFace model
df_hug_en = encode_categorical(df_hug)
df_hug_en.head()

In [ ]:
print(df_hug_en.shape)

#### What all categorical encoding techniques have you used & why did you use those techniques?

Our function converts categorical variables into numerical formats suitable for machine learning models through several techniques:

* **One-Hot Encoding for traveller_type:**
Creates binary columns for each traveller category
Uses drop_first=True to avoid the dummy variable trap
Example: "Business", "Family", "Solo" travelers each get their own column
* **Ordinal Encoding for cabin:**
Converts cabin classes to ordered numerical values based on their hierarchy
Economy (0) → Premium Economy (1) → Business (2) → First Class (3)
Preserves the natural ordering relationship between classes
* **Cleanup and Conversion:**
Removes original text-based categorical columns
Ensures proper data types (integers for cabin_level)
* **Target Variable Encoding:**
Transforms 'recommended' from text ('yes'/'no') to binary (1/0)
Creates proper format for binary classification models

This preprocessing step is crucial as most machine learning algorithms require numerical inputs while still preserving the underlying information in the categorical variables.



### 5. Feature Manipulation

#### 5.1. Feature Manipulation: Sentiment analysis using hugging face lib.

##### **df_hug_en**: For this dataset, we will use the roberta-lib. trained on twitter comments to tokenize and perform sentiment analysis on customer reviews.
                    - each review will be given 3 scores, i.e. probability of the pre-trained to determine whether a review is positive, neutral, or negative.
                    - https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment
                    - We will create 4 new features, namely,  neg_scores,  neu_scores,  pos_scores, and senti_final. We will then use these features to train and test our models.
                    - This is just a concept what we can do with the ever increasing A.I. models/tools we have in our disposal.

    Note: The time taken to tokenize and perform sentiment analysis on this dataset(~4,500) reviews is 25-30 mins.


In [ ]:
from transformers import AutoModelForSequenceClassification
from transformers import TFAutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoConfig
import numpy as np
from scipy.special import softmax

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Initializing the MODEL
MODEL = f"cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
config = AutoConfig.from_pretrained(MODEL)
# Using pytorch tensors(pt)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)
tokenizer.save_pretrained(MODEL)
model.save_pretrained(MODEL)

In [ ]:
"""The above warning is okay, as we are using the roberta model on another task"""
def get_tensor_size(text):
  tokenized = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
  tensor_size = tokenized['input_ids'].size()
  return tensor_size[1]

In [ ]:
df_hug_en.head()

In [ ]:
# Manipulate Features to minimize feature correlation and create new features
df_hug_en['tensor_size'] = df_hug_en['customer_review_clean'].apply(lambda x: get_tensor_size(x))
print(f"Dataset size: {df_hug_en.shape[0]}")
# creating a new dataframe where "customer_review_clean" <= 514
df_hugModel = df_hug_en.query("tensor_size <= 514")
print(f"New Dataset size: {df_hugModel.shape[0]}")
df_hugModel = df_hugModel.reset_index(inplace = False, drop=True)

In [ ]:
def pre_trained_sentiment_analysis(df):
  """
  EXECUTION TIME FOR THIS FUNC. IN COLLAB(FREE) IS ~40-50 mins!
  """
  # initializing list to store sentiment scores
  neg_scores = []
  neu_scores = []
  pos_scores = []

  # O(n): going through every row in the list in an orderly fashion!
  for text in df['customer_review_clean']:
    encoded_input = tokenizer(text, return_tensors='pt')
    output = model(**encoded_input)
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)
    neg_scores.append(scores[0])
    neu_scores.append(scores[1])
    pos_scores.append(scores[2])

  df['sentiment_neg'] = neg_scores
  df['sentiment_neu'] = neu_scores
  df['sentiment_pos'] = pos_scores

  return df

In [ ]:
# creating new features(sentiment scores)
df_hug_new = pre_trained_sentiment_analysis(df_hugModel)

In [ ]:
df_hug_new.head(5)

In [ ]:
df_hug_new = df_hug_new.drop(['customer_review_clean', 'tensor_size'], axis=1)
df_hug_new = df_hug_new.reset_index(inplace = False, drop=True)
df_hug_new.head(5)

In [ ]:
print(df_hug_new.shape)

In [ ]:
df_hug_new.skew()

#### 5.2. Feature Manipulation: Multi-collinearity check

##### **df_one_en**:

In [ ]:
# Manipulate Features to minimize feature correlation and create new features
df_one_en.head()

In [ ]:
# We will use this dataset for our decision based/tree models
# we will not perform any transformation on this dataset as those dataset won't be based on distance methods
df_one_base = df_one_en.copy()

In [ ]:
df_one_base.head()

In [ ]:
# We will use this dataset for logistic regression
# will convert most of the data to numeric features
# also try to remove multi-collinearity
df_one_reg = df_one_en.copy()

In [ ]:
df_one_reg.head()

In [ ]:
df_one_reg[['traveller_Couple Leisure',
            'traveller_Family Leisure',
            'traveller_Solo Leisure']] = df_one_reg[['traveller_Couple Leisure',
                                                     'traveller_Family Leisure',
                                                     'traveller_Solo Leisure']].astype(int)

In [ ]:
df_one_reg.head(5)

In [ ]:
correlation_matrix = df_one_reg[['overall',
                                 'seat_comfort',
                                 'cabin_service',
                                 'food_bev',
                                 'entertainment',
                                 'ground_service',
                                 'value_for_money',
                                 'traveller_Couple Leisure',
                                 'traveller_Family Leisure',
                                 'traveller_Solo Leisure',
                                 'cabin_level']].corr()

# Plot a heatmap to visualize correlations
plt.figure()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# Compute VIF for each feature
vif_data = pd.DataFrame()
vif_data["Feature"] = correlation_matrix.columns
vif_data["VIF"] = [variance_inflation_factor(correlation_matrix.values, i)
                    for i in range(correlation_matrix.shape[1])]

print(vif_data.sort_values(by="VIF", ascending=False))

##### **df_pmf_en**:

In [ ]:
df_pmf_en.head()

In [ ]:
df_pmf_base = df_pmf_en.copy()

In [ ]:
df_pmf_reg = df_pmf_en.copy()

In [ ]:
df_pmf_reg.head()

In [ ]:
df_pmf_reg[['traveller_Couple Leisure',
            'traveller_Family Leisure',
            'traveller_Solo Leisure']] = df_pmf_reg[['traveller_Couple Leisure',
                                                     'traveller_Family Leisure',
                                                     'traveller_Solo Leisure']].astype(int)

In [ ]:
df_pmf_reg.head(10)

In [ ]:
correlation_matrix = df_pmf_reg[['overall',
                                 'seat_comfort',
                                 'cabin_service',
                                 'food_bev',
                                 'entertainment',
                                 'ground_service',
                                 'value_for_money',
                                 'traveller_Couple Leisure',
                                 'traveller_Family Leisure',
                                 'traveller_Solo Leisure',
                                 'cabin_level']].corr()

# Plot a heatmap to visualize correlations
plt.figure()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# Compute VIF for each feature
vif_data = pd.DataFrame()
vif_data["Feature"] = correlation_matrix.columns
vif_data["VIF"] = [variance_inflation_factor(correlation_matrix.values, i)
                    for i in range(correlation_matrix.shape[1])]

print(vif_data.sort_values(by="VIF", ascending=False))

##### **df_hug_new:**

In [ ]:
df_hug_new.head()

In [ ]:
df_hug_reg = df_hug_new.copy()

In [ ]:
df_hug_reg.head()

In [ ]:
df_hug_reg[['traveller_Couple Leisure',
            'traveller_Family Leisure',
            'traveller_Solo Leisure']] = df_hug_reg[['traveller_Couple Leisure',
                                                     'traveller_Family Leisure',
                                                     'traveller_Solo Leisure']].astype(int)

In [ ]:
df_hug_reg.head(10)

In [ ]:
correlation_matrix = df_hug_reg[['overall',
                                 'seat_comfort',
                                 'cabin_service',
                                 'food_bev',
                                 'entertainment',
                                 'ground_service',
                                 'value_for_money',
                                 'traveller_Couple Leisure',
                                 'traveller_Family Leisure',
                                 'traveller_Solo Leisure',
                                 'cabin_level',
                                 'sentiment_neg',
                                 'sentiment_neu',
                                 'sentiment_pos']].corr()

# Plot a heatmap to visualize correlations
plt.figure(figsize=(10, 5))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# Compute VIF for each feature
vif_data = pd.DataFrame()
vif_data["Feature"] = correlation_matrix.columns
vif_data["VIF"] = [variance_inflation_factor(correlation_matrix.values, i)
                    for i in range(correlation_matrix.shape[1])]

print(vif_data.sort_values(by="VIF", ascending=False))

#### 5.3. Feature Selection

As we can see from the last operation that the VIF for all of features are very high. This usually suggest we should drop those features. But without domain expertise, that may not be a wise decision.
Most of the features are independent in real life, i.e. a customer may have a great ground service experience, but may experience on-flight experience very bad.
for now, we will keep all our features, and make a decision/tree based model, where we will hopefully learn which are the better features using feature_importance method.

### 6. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

Dimensionality reduction techniques can lead to information loss. Since we have only 10 features in our dataset, the risk of overfitting is reduced because the model has fewer opportunities to fit noise in the data. Hence no need to apply dimensionality reduction techniques such as PCA.

### 7. Data Scaling

In [ ]:
# Scaling your data
# Normalizing data using MinMaxScaler
"""
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 1))  # Default range is (0, 1)
# Fit and transform the dataset
scaled_data = scaler.fit_transform(df_one_reg)
# Convert scaled data back to a DataFrame for better readability
df_one_reg_scaled = pd.DataFrame(scaled_data, columns=df_one_reg.columns)
"""
# we will directly use the min-max scaler in our models, instead of using it as a func.

##### Which method have you used to scale you data and why?

Min-Max scaling is chosen for several important reasons in this context:

- Preserves relationships within features: Unlike standardization (z-score), Min-Max scaling maintains the original distribution shape while compressing the range, which is particularly valuable for features that don't follow a normal distribution.

- Interpretability: Values between 0-1 are easily interpretable as "percentages of the maximum value," making feature importance more intuitive in some contexts.

- Handling outliers differently: Unlike standardization which can still produce extreme values for outliers, Min-Max scaling compresses all values to the same fixed range. This can be beneficial when you want to preserve the presence of outliers but limit their impact.

- Feature uniformity: When features have widely different scales (like rating systems of 1-5 alongside metrics that might range in thousands), Min-Max scaling creates uniformity without assuming any particular statistical distribution.

For our specific data that likely includes various rating scales and categorical variables that have been encoded, Min-Max scaling provides a consistent transformation that preserves the relative differences in customer ratings while making all features comparable for model training.

### 8. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.
from sklearn.model_selection import train_test_split

def split_data_train_test(df, test_size_ = 0.2, ran_state_given = None):
  # creating feature dataframe
  X = df.drop('recommended', axis=1)
  # creating target series
  y = df['recommended']
  # print basic sizes of the datasets
  print(f"Feature dataset size: {X.shape}")
  print(f"Label dataset size: {len(y)} \n")
  # randomizing state for train_test_split
  if ran_state_given != None:
    ran_state = ran_state_given
  else:
    ran_state = np.random.choice(np.arange(1, 51))
  print(f"random_state = {ran_state}")
  X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                      test_size=test_size_,
                                                      random_state=ran_state)
  # Size of features and label datasets
  print(f"Training feature dataset size: {X_train.shape}")
  print(f"Training label dataset size: {y_train.shape[0]} \n")
  print(f"Test feature dataset size: {X_test.shape[0]}")
  print(f"Test label target dataset size: {y_test.shape[0]} \n")
  # printing all the feature names
  print(f"features: {list(X_train.columns)}")

  return X, y, X_train, X_test, y_train, y_test

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

The percentage of target variable category are almost equal, so there is no need for handling class imbalance.

## ***6. ML Model Implementation***

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, make_scorer, f1_score, roc_auc_score,  recall_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, average_precision_score
from sklearn.metrics import classification_report
from sklearn.model_selection import learning_curve, StratifiedKFold, GridSearchCV, cross_val_score
from itertools import product
import shap

In [ ]:
def fit_model(X_train, y_train, X_test, y_test, model):
  # training the model
  model.fit(X_train, y_train)
  train_class_preds = model.predict(X_train)
  test_class_preds = model.predict(X_test)

  return train_class_preds, test_class_preds

In [ ]:
def evaluate_model(train_class_preds, y_train, test_class_preds, y_test, evaluation_return = False):
  print("\n"+"-"*50+"MODEL EVALUATION"+"-"*50)
  # scoring the accuracy of the model on the training dataset
  train_accuracy = accuracy_score(train_class_preds, y_train)
  # scoring the accuracy of the model on the test dataset
  test_accuracy = accuracy_score(test_class_preds, y_test)
  recall = recall_score(y_test, test_class_preds)
  # printing accuracies of the model
  print(f"The accuracy on train data is {train_accuracy:.5f}")
  print(f"The accuracy on test data is {test_accuracy:.5f}\n")
  # creating the classification report
  report_train = classification_report(y_train, train_class_preds)
  print("For Train dataset, classification report: ")
  print(report_train)
  report_test = classification_report(y_test, test_class_preds)
  print("For Test dataset, classification report: ")
  print(report_train)
  # visualizing confusion matric for both the train and test dataset
  cm_train = confusion_matrix(y_true=y_train, y_pred=train_class_preds)
  cm_test = confusion_matrix(y_true=y_test, y_pred=test_class_preds)
  # creating plots
  fig, axes = plt.subplots(ncols=2, figsize=(12, 3))
  sns.heatmap(pd.DataFrame(cm_train), annot=True, cmap="YlGnBu", fmt='g', ax=axes[0]).set(xlabel='Predicted label', ylabel='Actual label', title='Confusion matrix(Train dataset)')
  sns.heatmap(pd.DataFrame(cm_test), annot=True, cmap="YlGnBu", fmt='g', ax=axes[1]).set(xlabel='Predicted label', ylabel='Actual label', title='Confusion matrix(Test dataset)')
  if evaluation_return == True:
    return train_accuracy, test_accuracy, recall

In [ ]:
def roc_auc_curve(y_true, y_probs, model_name=""):
  fpr, tpr, thres = roc_curve(y_true, y_probs)
  auc_score = roc_auc_score(y_true, y_probs)

  plt.figure(figsize=(5, 5))
  plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc_score:.2f})', color='blue')
  plt.plot([0, 1], [0, 1], 'k--', label=model_name)
  plt.title(f'ROC-AUC Curve')
  plt.xlabel('False Positive Rate')
  plt.ylabel('True Positive Rate')
  plt.legend()
  plt.grid()
  plt.show()

In [ ]:
def plot_learning_curve(estimator, X, y, cv, scoring='accuracy', train_sizes=np.linspace(0.1, 1.0, 10)):
  """
  Plots the learning curve for a given model.

  Parameters:
  - estimator: The machine learning model (e.g., LogisticRegression).
  - X: Features.
  - y: Target labels.
  - cv: Number of cross-validation splits.
  - scoring: Metric for evaluation (default is 'accuracy').
  - train_sizes: Proportions of the training set to use.
  """
  train_sizes, train_scores, val_scores = learning_curve(estimator, X, y, cv=cv, scoring=scoring, train_sizes=train_sizes, n_jobs=-1)

  # Calculate mean and standard deviation of training and validation scores
  train_scores_mean = np.mean(train_scores, axis=1)
  train_scores_std = np.std(train_scores, axis=1)
  val_scores_mean = np.mean(val_scores, axis=1)
  val_scores_std = np.std(val_scores, axis=1)

  # Plot the learning curves
  plt.figure(figsize=(10, 6))
  plt.plot(train_sizes, train_scores_mean, label='Training Score', color='blue', marker='o')
  plt.fill_between(train_sizes, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std, alpha=0.1, color='blue')
  plt.plot(train_sizes, val_scores_mean, label='Validation Score', color='green', marker='o')
  plt.fill_between(train_sizes, val_scores_mean - val_scores_std, val_scores_mean + val_scores_std, alpha=0.1, color='green')

  plt.title('Learning Curve')
  plt.xlabel('Training Set Size')
  plt.ylabel(scoring.capitalize())
  plt.legend(loc='best')
  plt.grid()
  plt.show()

In [ ]:
# Accuracy Scores Dataframe
scores_df = pd.DataFrame(columns = ['Model: Dataset', 'Accuracy(Train)', 'Accuracy(after CV)', 'ROC AUC score', 'Recall'])

### **ML Model - 1: Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression

#### **Logistic Regression: Model 1 (for df_one_reg)**

##### ***Data Scaling***

In [ ]:
df_one_reg.head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 1))  # Default range is (0, 1)
# Fit and transform the dataset
scaled_data = scaler.fit_transform(df_one_reg)
# Convert scaled data back to a DataFrame for better readability
df_one_reg_scaled = pd.DataFrame(scaled_data, columns=df_one_reg.columns)

In [ ]:
df_one_reg_scaled.head()

##### ***Data Splitting and Model Selection***

In [ ]:
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_one_reg_scaled, ran_state_given = 34)
# create a logistic regression model
model_LoR = LogisticRegression()
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model=model_LoR)
evaluate_model(train_preds, y_train, test_preds, y_test)

In [ ]:
roc_auc_curve(y_test, test_preds, "Logistic Regression(Scaled)")

##### ***Learning Curves***

In [ ]:
plot_learning_curve(model_LoR, X_train, y_train, cv=5, scoring='accuracy')

In [ ]:
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=22)
plot_learning_curve(model_LoR, X_train, y_train, cv=stratified_cv, scoring='accuracy')

##### ***Parameter Tunning***

In [ ]:
param_grid = {
              'C': [0.01, 0.1, 1, 10, 100],  # Regularization strength
              'penalty': ['l1', 'l2'],       # Regularization type
              'solver': ['liblinear']        # Compatible solver
             }

grid_search = GridSearchCV(model_LoR, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)
print("Best Parameters:", grid_search.best_params_)
print("Best Accuracy:", grid_search.best_score_)

##### **Generalized Model**

In [ ]:
# picking the parameters giving the best results
log_model_b_para_one = grid_search.best_estimator_
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_one_reg_scaled, ran_state_given = 1)
# create new model with best parameters
model_LoR_one = log_model_b_para_one
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model=model_LoR_one)
evaluate_model(train_preds, y_train, test_preds, y_test)

##### **Cross-Validation**

In [ ]:
random_num = np.random.choice(np.arange(1, 201))
stratified_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_num)
# Define metrics for evaluation
metrics = {
           'accuracy': 'accuracy',
           'f1_score': make_scorer(f1_score),
           'roc_auc': 'roc_auc',
           'precision': make_scorer(precision_score),
           'recall': make_scorer(recall_score)
          }
# Perform cross-validation for the above metrics
cv_results = {}
for metric_name, metric in metrics.items():
  scores = cross_val_score(model_LoR_one, X, y, cv=stratified_kf, scoring=metric, n_jobs=-1)
  cv_results[metric_name] = scores
  print(f"{metric_name.capitalize()} Scores: {scores}")
  print(f"Mean {metric_name.capitalize()}: {scores.mean():.4f}")
  print(f"Standard Deviation: {scores.std():.4f}\n")

In [ ]:
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=56)
plot_learning_curve(model_LoR_one, X_train, y_train, cv=stratified_cv, scoring='accuracy')

##### **Feature Importance**

In [ ]:
# feature importance according to shap
explainer = shap.Explainer(model_LoR_one, X_train)
shap_values = explainer(X_train)
# Compute mean absolute SHAP values for feature importance
shap_importance = np.abs(shap_values.values).mean(axis=0)
feature_importance = np.abs(model_LoR_one.coef_[0])
feature_names = X_train.columns if hasattr(X_train, 'columns') else [f'Feature {i}' for i in range(X_train.shape[1])]

# Combine into a DataFrame
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'SHAP Importance': shap_importance,
    'Co-efficent based Importance': feature_importance,
}).sort_values(by='SHAP Importance', ascending=False)

print(importance_df)

##### **Final Model: Logistic Regression for df_one_reg**

In [ ]:
df_one_reg_scaled_removed = df_one_reg_scaled.drop(['cabin_level', 'traveller_Solo Leisure', 'entertainment', 'traveller_Couple Leisure', 'traveller_Family Leisure'], axis=1)
df_one_reg_scaled_removed.head()

In [ ]:
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_one_reg_scaled_removed, ran_state_given = 9)
# create new model with best parameters
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model=model_LoR_one)
train_accu, test_accu, recall = evaluate_model(train_preds, y_train, test_preds, y_test, evaluation_return = True)

In [ ]:
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)
plot_learning_curve(model_LoR_one, X_train, y_train, cv=stratified_cv, scoring='accuracy')

In [ ]:
random_num = np.random.choice(np.arange(1, 201))
stratified_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_num)
mean_accu = 0
mean_recall = 0
mean_roc_auc = 0
# Define metrics for evaluation
metrics = {
           'accuracy': 'accuracy',
           'f1_score': make_scorer(f1_score),
           'roc_auc': 'roc_auc',
           'precision': make_scorer(precision_score),
           'recall': make_scorer(recall_score)
          }
# Perform cross-validation for the above metrics
cv_results = {}
for metric_name, metric in metrics.items():
  scores = cross_val_score(model_LoR_one, X, y, cv=stratified_kf, scoring=metric, n_jobs=-1)
  cv_results[metric_name] = scores
  print(f"{metric_name.capitalize()} Scores: {scores}")
  print(f"Mean {metric_name.capitalize()}: {scores.mean():.4f}")
  print(f"Standard Deviation: {scores.std():.4f}\n")
  if metric_name == 'accuracy':
    mean_accu = scores.mean()
  if metric_name == 'roc_auc':
    mean_roc_auc = scores.mean()
  if metric_name == 'recall':
    mean_recall = scores.mean()

#### **Final scores for model 1**

In [ ]:
scores_df.loc[0] = ['Logistic Regression: df_one_reg', train_accu, mean_accu, mean_roc_auc, mean_recall]
scores_df

#### **Logistic Regression: Model 2 (for df_pmf_reg)**

##### **Data Scaling**

In [ ]:
df_pmf_reg.head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 1))  # Default range is (0, 1)
# Fit and transform the dataset
scaled_data = scaler.fit_transform(df_pmf_reg)
# Convert scaled data back to a DataFrame for better readability
df_pmf_reg_scaled = pd.DataFrame(scaled_data, columns=df_pmf_reg.columns)

In [ ]:
df_pmf_reg_scaled.head()

##### **Data spliting and Model selection**

In [ ]:
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_pmf_reg_scaled, ran_state_given = 44)
# choose a model
model_LoR = LogisticRegression()
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model=model_LoR)
evaluate_model(train_preds, y_train, test_preds, y_test)

In [ ]:
roc_auc_curve(y_test, test_preds, "Logistic Regression(Scaled)")

##### **Learining curves**

In [ ]:
plot_learning_curve(model_LoR, X_train, y_train, cv=15, scoring='accuracy')

In [ ]:
random_num = np.random.choice(np.arange(1, 201))
stratified_cv = StratifiedKFold(n_splits=15, shuffle=True, random_state=28)
plot_learning_curve(model_LoR, X_train, y_train, cv=stratified_cv, scoring='accuracy')

##### **Parameter tunning**

In [ ]:
param_grid = {
              'C': [0.01, 0.1, 1, 10, 100],  # Regularization strength
              'penalty': ['l1', 'l2'],       # Regularization type
              'solver': ['liblinear']        # Compatible solver
             }

grid_search = GridSearchCV(model_LoR, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)
print("Best Parameters:", grid_search.best_params_)
print("Best Accuracy:", grid_search.best_score_)

##### **Generalised model**

In [ ]:
log_model_b_para_pmf = grid_search.best_estimator_
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_pmf_reg_scaled, ran_state_given = 32)
# choose a model
model_LoR_pmf = log_model_b_para_pmf
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model=model_LoR_pmf)
evaluate_model(train_preds, y_train, test_preds, y_test)

##### **Cross validation**

In [ ]:
stratified_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=101)
# Define metrics for evaluation
metrics = {
           'accuracy': 'accuracy',
           'f1_score': make_scorer(f1_score),
           'roc_auc': 'roc_auc',
           'precision': make_scorer(precision_score),
           'recall': make_scorer(recall_score)
          }

# Perform cross-validation for each metric
cv_results = {}
for metric_name, metric in metrics.items():
  scores = cross_val_score(model_LoR_pmf, X, y, cv=stratified_kf, scoring=metric, n_jobs=-1)
  cv_results[metric_name] = scores
  print(f"{metric_name.capitalize()} Scores: {scores}")
  print(f"Mean {metric_name.capitalize()}: {scores.mean():.4f}")
  print(f"Standard Deviation: {scores.std():.4f}\n")

In [ ]:
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=72)
plot_learning_curve(model_LoR_pmf, X_train, y_train, cv=stratified_cv, scoring='accuracy')

##### **Feature Importance**

In [ ]:
explainer = shap.Explainer(model_LoR_pmf, X_train)
shap_values = explainer(X_train)
# Compute mean absolute SHAP values for feature importance
shap_importance = np.abs(shap_values.values).mean(axis=0)
feature_importance = np.abs(model_LoR_pmf.coef_[0])
feature_names = X_train.columns if hasattr(X_train, 'columns') else [f'Feature {i}' for i in range(X_train.shape[1])]

# Combine into a DataFrame
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'SHAP Importance': shap_importance,
    'Co-efficent based Importance': feature_importance,
}).sort_values(by='SHAP Importance', ascending=False)

print(importance_df)

##### **Final Model: Logistic Regression for df_pmf_reg**

In [ ]:
df_pmf_reg_scaled_removed = df_pmf_reg_scaled.drop(['cabin_level', 'traveller_Solo Leisure', 'traveller_Couple Leisure', 'traveller_Family Leisure', 'seat_comfort'], axis=1)
df_pmf_reg_scaled_removed.head()

In [ ]:
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_pmf_reg_scaled_removed, ran_state_given = 47)
# create new model with best parameters
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model=model_LoR_pmf)
train_accu, test_accu, recall = evaluate_model(train_preds, y_train, test_preds, y_test, evaluation_return = True)

In [ ]:
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=83)
plot_learning_curve(model_LoR_pmf, X_train, y_train, cv=stratified_cv, scoring='accuracy')

In [ ]:
random_num = np.random.choice(np.arange(1, 201))
stratified_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_num)
mean_accu = 0
mean_recall = 0
mean_roc_auc = 0
# Define metrics for evaluation
metrics = {
           'accuracy': 'accuracy',
           'f1_score': make_scorer(f1_score),
           'roc_auc': 'roc_auc',
           'precision': make_scorer(precision_score),
           'recall': make_scorer(recall_score)
          }
# Perform cross-validation for the above metrics
cv_results = {}
for metric_name, metric in metrics.items():
  scores = cross_val_score(model_LoR_pmf, X, y, cv=stratified_kf, scoring=metric, n_jobs=-1)
  cv_results[metric_name] = scores
  print(f"{metric_name.capitalize()} Scores: {scores}")
  print(f"Mean {metric_name.capitalize()}: {scores.mean():.4f}")
  print(f"Standard Deviation: {scores.std():.4f}\n")
  if metric_name == 'accuracy':
    mean_accu = scores.mean()
  if metric_name == 'roc_auc':
    mean_roc_auc = scores.mean()
  if metric_name == 'recall':
    mean_recall = scores.mean()

#### **Final Scores for Model 2**

In [ ]:
scores_df.loc[1] = ['Logistic Regression: df_pmf_reg', train_accu, mean_accu, mean_roc_auc, mean_recall]
scores_df

#### **Logistic Regression: Model 3 (for df_hug_reg)**

##### ***Data Scaling***

In [ ]:
df_hug_reg.head()

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))  # Default range is (0, 1)
# Fit and transform the dataset
scaled_data = scaler.fit_transform(df_hug_reg)
# Convert scaled data back to a DataFrame for better readability
df_hug_reg_scaled = pd.DataFrame(scaled_data, columns=df_hug_reg.columns)

##### ***Data splitting and Model selection***

In [ ]:
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_hug_reg_scaled)
# choose a model
model_LoR = LogisticRegression()
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model=model_LoR)
evaluate_model(train_preds, y_train, test_preds, y_test)

In [ ]:
roc_auc_curve(y_test, test_preds, "Logistic Regression(Scaled)")

##### ***Learning Curves***

In [ ]:
plot_learning_curve(model_LoR, X_train, y_train, cv=10, scoring='accuracy')

In [ ]:
stratified_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=888)
plot_learning_curve(model_LoR, X_train, y_train, cv=stratified_cv, scoring='accuracy')

##### ***Parameter Tunning***

In [ ]:
param_grid = {
              'C': [0.01, 0.1, 1, 10, 100],  # Regularization strength
              'penalty': ['l1', 'l2'],       # Regularization type
              'solver': ['liblinear']        # Compatible solver
             }

grid_search = GridSearchCV(model_LoR, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)
print("Best Parameters:", grid_search.best_params_)
print("Best Accuracy:", grid_search.best_score_)

##### ***Generalized Model***

In [ ]:
best_log_model_b_para_hug = grid_search.best_estimator_
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_hug_reg_scaled, ran_state_given = 909)
# choose a model
model_LoR_hug = best_log_model_b_para_hug
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model=model_LoR_hug)
evaluate_model(train_preds, y_train, test_preds, y_test)

##### ***Cross-Validation***

In [ ]:
stratified_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=75)
# Define metrics for evaluation
metrics = {
           'accuracy': 'accuracy',
           'f1_score': make_scorer(f1_score),
           'roc_auc': 'roc_auc',
           'precision': make_scorer(precision_score),
           'recall': make_scorer(recall_score)
          }
# Perform cross-validation for each metric
cv_results = {}
for metric_name, metric in metrics.items():
  scores = cross_val_score(model_LoR_hug, X, y, cv=stratified_kf, scoring=metric, n_jobs=-1)
  cv_results[metric_name] = scores
  print(f"{metric_name.capitalize()} Scores: {scores}")
  print(f"Mean {metric_name.capitalize()}: {scores.mean():.4f}")
  print(f"Standard Deviation: {scores.std():.4f}\n")

##### ***Feature Importance and Selection***

In [ ]:
explainer = shap.Explainer(model_LoR_hug, X_train)  # Replace model_LoR with your model
shap_values = explainer(X_train)
# Compute mean absolute SHAP values for feature importance
shap_importance = np.abs(shap_values.values).mean(axis=0)
feature_importance = np.abs(model_LoR_hug.coef_[0])
feature_names = X_train.columns if hasattr(X_train, 'columns') else [f'Feature {i}' for i in range(X_train.shape[1])]

# Combine into a DataFrame
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'SHAP Importance': shap_importance,
    'Co-efficent based Importance': feature_importance,
}).sort_values(by='SHAP Importance', ascending=False)

print(importance_df)

##### ***Final Model for Logistic Regression for df_hug_reg***

In [ ]:
df_hug_reg_scaled_removed = df_hug_reg_scaled.drop(['cabin_level', 'traveller_Solo Leisure', 'entertainment', 'food_bev', 'sentiment_neu'], axis=1)
df_hug_reg_scaled_removed.head()

In [ ]:
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_hug_reg_scaled_removed, ran_state_given = 67)
# create new model with best parameters
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model=model_LoR_hug)
train_accu, test_accu, recall = evaluate_model(train_preds, y_train, test_preds, y_test, evaluation_return = True)

In [ ]:
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=86)
plot_learning_curve(model_LoR_hug, X_train, y_train, cv=stratified_cv, scoring='accuracy')

In [ ]:
random_num = np.random.choice(np.arange(1, 201))
stratified_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_num)
mean_accu = 0
mean_recall = 0
mean_roc_auc = 0
# Define metrics for evaluation
metrics = {
           'accuracy': 'accuracy',
           'f1_score': make_scorer(f1_score),
           'roc_auc': 'roc_auc',
           'precision': make_scorer(precision_score),
           'recall': make_scorer(recall_score)
          }
# Perform cross-validation for the above metrics
cv_results = {}
for metric_name, metric in metrics.items():
  scores = cross_val_score(model_LoR_hug, X, y, cv=stratified_kf, scoring=metric, n_jobs=-1)
  cv_results[metric_name] = scores
  print(f"{metric_name.capitalize()} Scores: {scores}")
  print(f"Mean {metric_name.capitalize()}: {scores.mean():.4f}")
  print(f"Standard Deviation: {scores.std():.4f}\n")
  if metric_name == 'accuracy':
    mean_accu = scores.mean()
  if metric_name == 'roc_auc':
    mean_roc_auc = scores.mean()
  if metric_name == 'recall':
    mean_recall = scores.mean()

#### ***Final Scores for Model 3***

In [ ]:
scores_df.loc[2] = ['Logistic Regression: df_hug_reg', train_accu, mean_accu, mean_roc_auc, mean_recall]
scores_df

#### Explain the ML Model used and it's performance using Evaluation metric Score Chart.

For our first model, I have choosen Logistic Regression(as one of the most popular classification model). Based on our 3 prepared datatsets, the model is first fitted un-filtered, and then according to the best hypertunned parameters, and finally feature selection for all 3 datasets.

- **Data Preparation and Scaling:**
  - We are using Min-Max scaling to normalize your features to a 0-1 range
  - This ensures all features contribute equally to the model regardless of their original scales


- **Initial Model Training and Evaluation:**
  - Basic logistic regression model to establish a baseline
  - Performance evaluation including ROC curve analysis
  - Learning curve plotting to assess model learning behavior


- **Hyperparameter Optimization:**
  - Grid search with cross-validation to find optimal parameters
  - Testing various regularization strengths (C values) and penalty types (L1, L2)
  - Using the liblinear solver which works well with both regularization types
  - Results in best parameters being identified for your specific dataset


- **Model Interpretation:**
  - SHAP analysis to understand feature importance in a model-agnostic way
  - Coefficient-based importance specific to logistic regression
  - Creating a ranked feature importance dataframe to identify key predictors


- **Robust Validation:**
  - Stratified k-fold cross-validation with 5 splits to ensure reliable performance estimation
  - Comprehensive metrics suite: accuracy, F1, ROC AUC, precision, and recall
  - Random state variation to ensure results aren't dependent on a specific data split

The **PMF-based imputation** strategy proved superior because it preserves the relationship between features and the target variable when handling missing values. Unlike traditional imputation methods that use generic statistics like mean or mode, the PMF approach uses conditional probabilities based on the target class, essentially saying: "What value would this feature likely have, given that the customer recommended (or didn't recommend) the service?"

Performance:
- The probability mass function (PMF) approach to handling missing values clearly provides the most predictive power, likely because it preserves more information by considering the relationship between features and the target variable.
- The sentiment analysis features from the Hugging Face model contribute to the highest recall, suggesting that sentiment extracted from reviews helps capture positive recommendations that might be missed by rating features alone.
- All models perform exceptionally well (>95% on all metrics), indicating that any would be suitable for production use, with your choice depending on whether you prioritize overall accuracy (PMF model) or recall (Hugging Face model).
- The very small difference between training and cross-validation accuracy across all models suggests good generalization without overfitting.

#### Which hyperparameter optimization technique have you used and why?

The reasons for choosing Grid Search include:

- **Exhaustive search**: Grid Search systematically works through every combination of the parameters you specified (C and penalty values), ensuring you find the optimal configuration.
- **Cross-validation integration**: By setting cv=5, we're using 5-fold cross-validation for each parameter combination, which helps prevent overfitting to your training data.
- **Appropriate for smaller parameter spaces:** With only 10 combinations to evaluate (5 C values × 2 penalty types), Grid Search is computationally feasible and efficient.
- **Deterministic results:** Unlike randomized methods, Grid Search provides deterministic, reproducible results by testing every parameter combination.
- **Parallelization:** You've set n_jobs=-1 to utilize all available CPU cores, making the search process more efficient.

This approach is particularly well-suited for logistic regression, where the parameter space is relatively small and the training process is fast enough to allow for an exhaustive search without excessive computational cost.

#### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The most significant improvement came not from hyperparameter tuning **but from the superior data preparation strategy (PMF-based imputation)**. The PMF approach to handling missing values provided approximately a **1% improvement across** **all metrics**, which is substantial considering the already high baseline performance.
The **hyperparameter optimization** likely contributed to **model stability** rather than dramatic performance increases, as evidenced by the low standard deviations in your cross-validation metrics.
This analysis demonstrates that for your classification task, the choice of data preparation strategy had a greater impact than model tuning, highlighting the importance of thoughtful feature engineering and imputation strategies. It can also be said that the hugging face model might have performed even better than the PMF model, but **due to limited resources**, it could only be trained in ~5k training dataset. **Further, sentiment analysis combined with PMF dataset may have been given more superior results**


### **ML Model - 2: Random Forest Classifier**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

#### **Random Forest Classifier: Model 4 (for df_one_reg)**

##### ***Data splitting and Model selection***

In [ ]:
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_one_reg, ran_state_given = 3)
# creating a Random Forest Classifier Model
model_RFM = RandomForestClassifier()
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model=model_RFM)
evaluate_model(train_preds, y_train, test_preds, y_test)

In [ ]:
roc_auc_curve(y_test, test_preds, "Random Forest Model(Dataset 1)")

##### ***Learning curve***

In [ ]:
plot_learning_curve(model_RFM, X_train, y_train, cv=10, scoring='accuracy')

There is overfitting in the above

##### ***Hyper-parameters tunning***

In [ ]:
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_one_reg, ran_state_given = 16)
# defining model hyper-parameters
param_grid1 = {
               'n_estimators': [50, 100, 200],         # Number of trees
               'max_depth': [2, 5, 7],                 # Limit tree depth
               'min_samples_split': [2, 5, 10],        # Minimum samples required to split an internal node
               'min_samples_leaf': [1, 2, 4]           # Minimum samples required at a leaf node
              }
# Create all parameter combinations
param_combinations = list(product(
                                  param_grid1['n_estimators'],
                                  param_grid1['max_depth'],
                                  param_grid1['min_samples_split'],
                                  param_grid1['min_samples_leaf'],
                                 )
                         )
print(f"Total combinations to evaluate: {len(param_combinations)}")

In [ ]:
best_diff = 100
b_n_estimators = None
b_max_depth = None
b_min_samples_split = None
b_min_samples_leaf = None

# iterating over all the combinations to select the model with least difference between train and test scores
for params in param_combinations:
  n_estimators, max_depth, min_samples_split, min_samples_leaf = params
  # create a new RFM to find the best hyper-paramater
  rf_model = RandomForestClassifier(n_estimators=n_estimators,
                                    max_depth=max_depth,
                                    min_samples_split=min_samples_split,
                                    min_samples_leaf=min_samples_leaf,
                                    )
  # fit the model
  rf_model.fit(X_train, y_train)
  print(f"for {n_estimators}, {max_depth}, {min_samples_split}, {min_samples_leaf}")
  # make train and test predictions
  train_class_preds = rf_model.predict(X_train)
  test_class_preds = rf_model.predict(X_test)
  # get train accuracy score
  train_accuracy = accuracy_score(train_class_preds, y_train)
  # get test accuracy score
  # print train accuracy score
  print(f"The accuracy on train data is {train_accuracy:.5f}")
  test_accuracy = accuracy_score(test_class_preds, y_test)
  # scoring the accuracy of the model on the test dataset
  print(f"The accuracy on test data is {test_accuracy:.5f}")
  curr_diff = abs(train_accuracy - test_accuracy)
  print(f"current diff: {curr_diff}")
  if curr_diff < best_diff:
    best_diff = curr_diff
    b_n_estimators = n_estimators
    b_max_depth = max_depth
    b_min_samples_split = min_samples_split
    b_min_samples_leaf = min_samples_leaf
  print(f"best_diff: {best_diff}")
  print("\n")

print(f"The least difference in accuracy(b/w train and test) was found to be: {best_diff} with the paramters:")
print(f"n_estimators: {b_n_estimators}, max_depth: {b_max_depth}, min_samples_split: {b_min_samples_split}, min_samples_leaf: {b_min_samples_leaf}")

##### ***Genralized model***

In [ ]:
# taking the hyper-parameters with the least train and test difference
print(f"for n_estimators: {b_n_estimators}, max_depth: {b_max_depth}, min_samples_split: {b_min_samples_split}, min_samples_leaf: {min_samples_leaf}")
# split data and train a new model
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_one_reg, ran_state_given = 17)
# creating a tuned model
model_RFM_tuned_one = RandomForestClassifier(
                                             n_estimators = b_n_estimators,
                                             max_depth = b_max_depth,
                                             min_samples_split = b_min_samples_split,
                                             min_samples_leaf = b_min_samples_leaf,
                                            )
# make predictions on train & test datasets
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model = model_RFM_tuned_one)
evaluate_model(train_preds, y_train, test_preds, y_test)

In [ ]:
roc_auc_curve(y_test, test_preds, "RFM")

##### ***Learning curves(for generalized/tunned model)***

In [ ]:
plot_learning_curve(model_RFM_tuned_one, X_train, y_train, cv=10, scoring='accuracy')

In [ ]:
stratified_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=10)
plot_learning_curve(model_RFM_tuned_one, X_train, y_train, cv=stratified_cv, scoring='accuracy')

##### ***Cross-validation***

In [ ]:
random_num = ran_state = np.random.choice(np.arange(1, 201))
stratified_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_num)
# Define metrics for evaluation
metrics = {
           'accuracy': 'accuracy',
           'f1_score': make_scorer(f1_score),
           'roc_auc': 'roc_auc',
           'precision': make_scorer(precision_score),
           'recall': make_scorer(recall_score)
          }
# Perform cross-validation for each metric
cv_results = {}
for metric_name, metric in metrics.items():
  scores = cross_val_score(model_RFM_tuned_one, X, y, cv=stratified_kf, scoring=metric, n_jobs=-1)
  cv_results[metric_name] = scores
  print(f"{metric_name.capitalize()} Scores: {scores}")
  print(f"Mean {metric_name.capitalize()}: {scores.mean():.4f}")
  print(f"Standard Deviation: {scores.std():.4f}\n")

##### ***Feature Importance***

In [ ]:
feature_importance = model_RFM_tuned_one.feature_importances_
feature_names = X_train.columns
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": feature_importance
}).sort_values(by="Importance", ascending=False)

print(importance_df)
importance_df.plot(kind="bar", x="Feature", y="Importance", figsize=(10, 6), title="Feature Importance")

In [ ]:
df_one_reg_rmf_fea = df_one_reg.drop(['cabin_level', 'traveller_Couple Leisure', 'traveller_Family Leisure', 'traveller_Solo Leisure', 'entertainment', 'food_bev',	'ground_service'], axis=1)
df_one_reg_rmf_fea.head()

##### ***Final Model: Random Forest Classifier for df_one_reg***

In [ ]:
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_one_reg_rmf_fea, ran_state_given = 1)
# creating a Random Forest Classifier Model
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model=model_RFM_tuned_one)
train_accu, test_accu, recall = evaluate_model(train_preds, y_train, test_preds, y_test, evaluation_return = True)

In [ ]:
stratified_cv = StratifiedKFold(n_splits=7, shuffle=True, random_state=85)
plot_learning_curve(model_RFM_tuned_one, X_train, y_train, cv=stratified_cv, scoring='accuracy')

In [ ]:
stratified_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=22)
mean_accu = 0
mean_recall = 0
mean_roc_auc = 0
# Define metrics for evaluation
metrics = {
           'accuracy': 'accuracy',
           'f1_score': make_scorer(f1_score),
           'roc_auc': 'roc_auc',
           'precision': make_scorer(precision_score),
           'recall': make_scorer(recall_score)
          }
# Perform cross-validation for the above metrics
cv_results = {}
for metric_name, metric in metrics.items():
  scores = cross_val_score(model_RFM_tuned_one, X, y, cv=stratified_kf, scoring=metric, n_jobs=-1)
  cv_results[metric_name] = scores
  print(f"{metric_name.capitalize()} Scores: {scores}")
  print(f"Mean {metric_name.capitalize()}: {scores.mean():.4f}")
  print(f"Standard Deviation: {scores.std():.4f}\n")
  if metric_name == 'accuracy':
    mean_accu = scores.mean()
  if metric_name == 'roc_auc':
    mean_roc_auc = scores.mean()
  if metric_name == 'recall':
    mean_recall = scores.mean()

#### ***Final scores for model 4***

In [ ]:
scores_df.loc[3] = ['Random Forest Classifier: df_one_reg', train_accu, mean_accu, mean_roc_auc, mean_recall]
scores_df

#### **Random Forest Classifier: Model 5 (for df_pmf_reg)**

##### ***Data splitting and Model selection***

In [ ]:
df_pmf_reg.head()

In [ ]:
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_pmf_reg, ran_state_given=7)
# choose a model
model_RMF = RandomForestClassifier()
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model=model_RMF)
evaluate_model(train_preds, y_train, test_preds, y_test)

In [ ]:
roc_auc_curve(y_test, test_preds, "RFM")

##### ***Learning curve***

In [ ]:
plot_learning_curve(model_RMF, X_train, y_train, cv=5, scoring='accuracy')

##### **Hyper-parameter tunning**

In [ ]:
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_pmf_reg, ran_state_given=6)
# defining model hyper-parameters
param_grid = {
              'n_estimators': [50, 100, 200],          # Number of trees
              'max_depth': [2, 5, 7],                  # Limit tree depth
              'min_samples_split': [2, 5, 10],         # Minimum samples required to split an internal node
              'min_samples_leaf': [1, 2, 4]            # Minimum samples required at a leaf node
             }
# Create all parameter combinations
param_combinations = list(product(
                                  param_grid['n_estimators'],
                                  param_grid['max_depth'],
                                  param_grid['min_samples_split'],
                                  param_grid['min_samples_leaf']
                                  )
                         )
print(f"Total combinations to evaluate: {len(param_combinations)}")

In [ ]:
best_diff = 100
b_n_estimators = None
b_max_depth = None
b_min_samples_split = None
b_min_samples_leaf = None
# iterating over all combinations to find the
for params in param_combinations:
  n_estimators, max_depth, min_samples_split, min_samples_leaf = params
  # iterating over all the combinations to select the model with least difference between train and test scores
  rf_model = RandomForestClassifier(n_estimators=n_estimators,
                                    max_depth=max_depth,
                                    min_samples_split=min_samples_split,
                                    min_samples_leaf=min_samples_leaf)
  # fit the model
  rf_model.fit(X_train, y_train)
  print(f"for {n_estimators}, {max_depth}, {min_samples_split}, {min_samples_leaf}")
  # make train and test predictions
  train_class_preds = rf_model.predict(X_train)
  test_class_preds = rf_model.predict(X_test)
  # get train accuracy score
  train_accuracy = accuracy_score(train_class_preds, y_train)
  # get test accuracy score
  # print train accuracy score
  print(f"The accuracy on train data is {train_accuracy:.5f}")
  test_accuracy = accuracy_score(test_class_preds, y_test)
  # scoring the accuracy of the model on the test dataset
  print(f"The accuracy on test data is {test_accuracy:.5f}")
  curr_diff = abs(train_accuracy - test_accuracy)
  print(f"current diff: {curr_diff}")
  if curr_diff < best_diff:
    best_diff = curr_diff
    b_n_estimators = n_estimators
    b_max_depth = max_depth
    b_min_samples_split = min_samples_split
    b_min_samples_leaf = min_samples_leaf
  print(f"best_diff: {best_diff}")
  print("\n")

print(f"The least difference in accuracy(b/w train and test) was found to be: {best_diff} with the paramters:")
print(f"n_estimators: {b_n_estimators}, max_depth: {b_max_depth}, min_samples_split: {b_min_samples_split}, min_samples_leaf: {b_min_samples_leaf}")

##### ***Generalized Model***

In [ ]:
print(f"for n_estimators: {b_n_estimators}, max_depth: {b_max_depth}, min_samples_split: {b_min_samples_split}, min_samples_leaf: {min_samples_leaf}")
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_pmf_reg)
# create model with tunned hyper-parameters
model_RFM_tuned_pmf = RandomForestClassifier(
                                             n_estimators=b_n_estimators,
                                             max_depth=b_max_depth,
                                             min_samples_split=b_min_samples_split,
                                             min_samples_leaf=b_min_samples_leaf
                                            )
# make predictions
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model = model_RFM_tuned_pmf)
evaluate_model(train_preds, y_train, test_preds, y_test)

In [ ]:
roc_auc_curve(y_test, test_preds, "RFM")

##### ***Learning curves(for Tunned/Generalized model)***

In [ ]:
plot_learning_curve(model_RFM_tuned_pmf, X_train, y_train, cv=10, scoring='accuracy')

In [ ]:
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=302)
plot_learning_curve(model_RFM_tuned_pmf, X_train, y_train, cv=stratified_cv, scoring='accuracy')

##### ***Cross-validation***

In [ ]:
stratified_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=35)
# Define metrics for evaluation
metrics = {
           'accuracy': 'accuracy',
           'f1_score': make_scorer(f1_score),
           'roc_auc': 'roc_auc',
           'precision': make_scorer(precision_score),
           'recall': make_scorer(recall_score)
          }
# Perform cross-validation for each metric
cv_results = {}
for metric_name, metric in metrics.items():
  scores = cross_val_score(model_RFM_tuned_pmf, X, y, cv=stratified_kf, scoring=metric, n_jobs=-1)
  cv_results[metric_name] = scores
  print(f"{metric_name.capitalize()} Scores: {scores}")
  print(f"Mean {metric_name.capitalize()}: {scores.mean():.4f}")
  print(f"Standard Deviation: {scores.std():.4f}\n")

##### ***Feature Importance***

In [ ]:
feature_importance = model_RFM_tuned_pmf.feature_importances_
feature_names = X_train.columns
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": feature_importance
}).sort_values(by="Importance", ascending=False)

print(importance_df)
importance_df.plot(kind="bar", x="Feature", y="Importance", figsize=(10, 6), title="Feature Importance")

In [ ]:
df_pmf_reg_rmf_fea = df_pmf_reg.drop(['cabin_level', 'traveller_Couple Leisure', 'traveller_Family Leisure', 'traveller_Solo Leisure', 'entertainment', 'food_bev',	'seat_comfort'], axis=1)
df_pmf_reg_rmf_fea.head()

##### ***Final Model: Random Forest Classifier for df_pmf_reg***

In [ ]:
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_pmf_reg_rmf_fea, ran_state_given = 21)
# creating a Random Forest Classifier Model
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model=model_RFM_tuned_pmf)
train_accu, test_accu, recall = evaluate_model(train_preds, y_train, test_preds, y_test, evaluation_return = True)

In [ ]:
stratified_cv = StratifiedKFold(n_splits=7, shuffle=True, random_state=85)
plot_learning_curve(model_RFM_tuned_pmf, X_train, y_train, cv=stratified_cv, scoring='accuracy')

In [ ]:
stratified_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=22)
mean_accu = 0
mean_recall = 0
mean_roc_auc = 0
# Define metrics for evaluation
metrics = {
           'accuracy': 'accuracy',
           'f1_score': make_scorer(f1_score),
           'roc_auc': 'roc_auc',
           'precision': make_scorer(precision_score),
           'recall': make_scorer(recall_score)
          }
# Perform cross-validation for the above metrics
cv_results = {}
for metric_name, metric in metrics.items():
  scores = cross_val_score(model_RFM_tuned_pmf, X, y, cv=stratified_kf, scoring=metric, n_jobs=-1)
  cv_results[metric_name] = scores
  print(f"{metric_name.capitalize()} Scores: {scores}")
  print(f"Mean {metric_name.capitalize()}: {scores.mean():.4f}")
  print(f"Standard Deviation: {scores.std():.4f}\n")
  if metric_name == 'accuracy':
    mean_accu = scores.mean()
  if metric_name == 'roc_auc':
    mean_roc_auc = scores.mean()
  if metric_name == 'recall':
    mean_recall = scores.mean()

#### ***Final scores for model 5***

In [ ]:
scores_df.loc[4] = ['Random Forest Classifier: df_pmf_reg', train_accu, mean_accu, mean_roc_auc, mean_recall]
scores_df

#### **Random Forest Classifier: Model 6 (for df_hug_reg)**

##### ***Data Splitting and Model Selection***

In [ ]:
df_hug_reg.head()

In [ ]:
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_hug_reg)
# choose a model
model_RMF = RandomForestClassifier()
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model=model_RMF)
evaluate_model(train_preds, y_train, test_preds, y_test)

In [ ]:
roc_auc_curve(y_test, test_preds, "RFM")

##### ***Learning Curve***

In [ ]:
plot_learning_curve(model_RMF, X_train, y_train, cv=5, scoring='accuracy')

##### ***Hyper-parameter Tunning***

In [ ]:
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_hug_reg)
param_grid = {
              'n_estimators': [50, 100, 200],         # Number of trees
              'max_depth': [2, 5, 7],                 # Limit tree depth
              'min_samples_split': [2, 5, 10],        # Minimum samples required to split an internal node
              'min_samples_leaf': [1, 2, 4]           # Minimum samples required at a leaf node
              }

# Create all parameter combinations
param_combinations = list(product(param_grid['n_estimators'],
                                  param_grid['max_depth'],
                                  param_grid['min_samples_split'],
                                  param_grid['min_samples_leaf']
                                  )
                         )
print(f"Total combinations to evaluate: {len(param_combinations)}")

In [ ]:
best_diff = float('inf')
b_n_estimators = None
b_max_depth = None
b_min_samples_split = None
b_min_samples_leaf = None

for params in param_combinations:
  n_estimators, max_depth, min_samples_split, min_samples_leaf = params
  # create a new RFM to find the best hyper-paramater
  rf_model = RandomForestClassifier(n_estimators=n_estimators,
                                    max_depth=max_depth,
                                    min_samples_split=min_samples_split,
                                    min_samples_leaf=min_samples_leaf)
  # fit the model
  rf_model.fit(X_train, y_train)
  print(f"for {n_estimators}, {max_depth}, {min_samples_split}, {min_samples_leaf}")
  # make train and test predictions
  train_class_preds = rf_model.predict(X_train)
  test_class_preds = rf_model.predict(X_test)
  # get train accuracy score
  train_accuracy = accuracy_score(train_class_preds, y_train)
  # get test accuracy score
  # print train accuracy score
  print(f"The accuracy on train data is {train_accuracy:.5f}")
  test_accuracy = accuracy_score(test_class_preds, y_test)
  # scoring the accuracy of the model on the test dataset
  print(f"The accuracy on test data is {test_accuracy:.5f}")
  curr_diff = abs(train_accuracy - test_accuracy)
  print(f"current diff: {curr_diff}")
  if curr_diff < best_diff:
    best_diff = curr_diff
    b_n_estimators = n_estimators
    b_max_depth = max_depth
    b_min_samples_split = min_samples_split
    b_min_samples_leaf = min_samples_leaf
  print(f"best_diff: {best_diff}")
  print("\n")

print(f"The least difference in accuracy(b/w train and test) was found to be: {best_diff} with the paramters:")
print(f"n_estimators: {b_n_estimators}, max_depth: {b_max_depth}, min_samples_split: {b_min_samples_split}, min_samples_leaf: {b_min_samples_leaf}")

##### ***Generalized Model***

In [ ]:
print(f"for n_estimators: {b_n_estimators}, max_depth: {b_max_depth}, min_samples_split: {b_min_samples_split}, min_samples_leaf: {min_samples_leaf}")
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_hug_reg, ran_state_given=20)
# choose a model
model_RFM_tuned_hug = RandomForestClassifier(
                                             n_estimators = b_n_estimators,
                                             max_depth = b_max_depth,
                                             min_samples_split = b_min_samples_split,
                                             min_samples_leaf = b_min_samples_leaf
                                            )
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model = model_RFM_tuned_hug)
evaluate_model(train_preds, y_train, test_preds, y_test)

In [ ]:
roc_auc_curve(y_test, test_preds, "RFM")

##### ***Learning Curves(for Tunned/Generalized Model)***

In [ ]:
plot_learning_curve(model_RFM_tuned_hug, X_train, y_train, cv=10, scoring='accuracy')

In [ ]:
stratified_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=24)
plot_learning_curve(model_RFM_tuned_hug, X_train, y_train, cv=stratified_cv, scoring='accuracy')

##### ***Cross-validation***

In [ ]:
stratified_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=22)
# Define metrics for evaluation
metrics = {
           'accuracy': 'accuracy',
           'f1_score': make_scorer(f1_score),
           'roc_auc': 'roc_auc',
           'precision': make_scorer(precision_score),
           'recall': make_scorer(recall_score)
          }
# Perform cross-validation for the above metrics
cv_results = {}
for metric_name, metric in metrics.items():
  scores = cross_val_score(model_RFM_tuned_hug, X, y, cv=stratified_kf, scoring=metric, n_jobs=-1)
  cv_results[metric_name] = scores
  print(f"{metric_name.capitalize()} Scores: {scores}")
  print(f"Mean {metric_name.capitalize()}: {scores.mean():.4f}")
  print(f"Standard Deviation: {scores.std():.4f}\n")

##### ***Feature Importance***

In [ ]:
feature_importance = model_RFM_tuned_hug.feature_importances_
feature_names = X_train.columns
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": feature_importance
}).sort_values(by="Importance", ascending=False)

print(importance_df)
importance_df.plot(kind="bar", x="Feature", y="Importance", figsize=(10, 6), title="Feature Importance")

In [ ]:
df_hug_reg_rmf_fea = df_hug_reg.drop(['cabin_level', 'traveller_Couple Leisure', 'traveller_Family Leisure', 'traveller_Solo Leisure', 'entertainment', 'food_bev',	'ground_service', 'sentiment_neu', 'food_bev'], axis=1)
df_hug_reg_rmf_fea.head()

##### ***Final Model: Random Forest Classifier for df_hug_reg***

In [ ]:
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_hug_reg_rmf_fea, ran_state_given = 777)
# creating a Random Forest Classifier Model
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model=model_RFM_tuned_hug)
train_accu, test_accu, recall = evaluate_model(train_preds, y_train, test_preds, y_test, evaluation_return = True)

In [ ]:
stratified_cv = StratifiedKFold(n_splits=7, shuffle=True, random_state=85)
plot_learning_curve(model_RFM_tuned_hug, X_train, y_train, cv=stratified_cv, scoring='accuracy')

In [ ]:
stratified_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=22)
mean_accu = 0
mean_recall = 0
mean_roc_auc = 0
# Define metrics for evaluation
metrics = {
           'accuracy': 'accuracy',
           'f1_score': make_scorer(f1_score),
           'roc_auc': 'roc_auc',
           'precision': make_scorer(precision_score),
           'recall': make_scorer(recall_score)
          }
# Perform cross-validation for the above metrics
cv_results = {}
for metric_name, metric in metrics.items():
  scores = cross_val_score(model_RFM_tuned_hug, X, y, cv=stratified_kf, scoring=metric, n_jobs=-1)
  cv_results[metric_name] = scores
  print(f"{metric_name.capitalize()} Scores: {scores}")
  print(f"Mean {metric_name.capitalize()}: {scores.mean():.4f}")
  print(f"Standard Deviation: {scores.std():.4f}\n")
  if metric_name == 'accuracy':
    mean_accu = scores.mean()
  if metric_name == 'roc_auc':
    mean_roc_auc = scores.mean()
  if metric_name == 'recall':
    mean_recall = scores.mean()

#### ***Final scores for model 6***

In [ ]:
scores_df.loc[5] = ['Random Forest Classifier: df_hug_reg', train_accu, mean_accu, mean_roc_auc, mean_recall]
scores_df

#### Explain the ML Model used and it's performance using Evaluation metric Score Chart.

We have choosen the second model to be a Random Forest Classifier, focusing on model development, optimization, and evaluation. Here's a summary:

- **Initial Model Setup:**
  - Splits data into training and test sets
  - Creates a baseline Random Forest classifier
  - Evaluates the model using ROC-AUC curve and learning curve visualization

- **Hyperparameter Tuning:**
  - Creates a grid of hyperparameters (n_estimators, max_depth, min_samples_split, min_samples_leaf)
  - Tests all combinations to find optimal parameters with minimum train-test accuracy difference
  - Tracks the best parameters through each iteration

- **Final Model Development:**
  - Creates a tuned Random Forest model with the optimal parameters
Evaluates performance on new train/test split

- **Cross-Validation:**
  - Implements 5-fold stratified cross-validation
  - Evaluates multiple metrics: accuracy, F1 score, ROC-AUC, precision, and recall

- **Feature Analysis:**
  - Extracts and visualizes feature importance
  -Creates a reduced dataset by removing less important features
  -Tests the tuned model on the reduced feature set

Here again, The **PMF-based imputation** strategy proved superior because it preserves the relationship between features and the target variable when handling missing values.

#### Which hyperparameter optimization technique have you used and why?

The hyperparameter optimization code implements a comprehensive grid search approach for tuning the Random Forest Classifier. The code systematically evaluates all possible combinations of four key parameters: n_estimators (50, 100, 200), max_depth (2, 5, 7), min_samples_split (2, 5, 10), and min_samples_leaf (1, 2, 4).
Rather than **optimizing for maximum accuracy alone, the code specifically targets the smallest difference between training and testing accuracy.** This strategy deliberately prioritizes model generalization over raw performance, helping to prevent overfitting. **By minimizing this gap, the resulting model is more likely to maintain consistent performance when deployed on new, unseen data.**
The implementation tracks the best parameter combination throughout the entire search process, ultimately selecting the configuration that produces the most balanced model. This approach demonstrates a focus on creating robust, reliable models rather than those that merely perform well on training data but may fail to generalize in real-world applications.

#### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Looking at these cross-validation results from our tuned Random Forest model, we can see again that these results validate our hyperparameter optimization strategy of minimizing the train-test accuracy gap. The model not only achieves high performance but does so consistently across different data partitions, demonstrating strong generalization capability. The low standard deviations across all metrics confirm that our focus on reducing overfitting has resulted in a robust model that should perform reliably on new, unseen data.

### **ML Model - 3: LightGBM Classifier**

In [ ]:
!pip install lightgbm

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV
import warnings

warnings.filterwarnings('ignore')

#### **LightGBM Classifier: Model 7(df_one_reg)**

##### ***Data Splitting and Model Selection***

In [ ]:
df_one_reg.head()

In [ ]:
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_one_reg, ran_state_given=29)
# choose a model
model_lgb = LGBMClassifier(metric="auc")
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model = model_lgb)
evaluate_model(train_preds, y_train, test_preds, y_test)

In [ ]:
roc_auc_curve(y_test, test_preds, "LightGBM(df_one_reg)")

##### ***Learning Curves***

In [ ]:
plot_learning_curve(model_lgb, X_train, y_train, cv=5, scoring='accuracy')

In [ ]:
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=25)
plot_learning_curve(model_lgb, X_train, y_train, cv=stratified_cv, scoring='accuracy')

##### ***Hyper-parameter Tunning***

In [ ]:
param_grid = {
              'num_leaves': [31, 50],
              'max_depth': [-1, 10],
              'learning_rate': [0.05, 0.1],
              'n_estimators': [100, 200],
              'min_child_samples': [20, 30]
             }

In [ ]:
lgbm = LGBMClassifier()
grid_search = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid,
    cv=3,               # 3-fold cross-validation
    scoring='accuracy', # Optimize for accuracy
    n_jobs=-1,          # Use all available CPU cores
    verbose=1           # Print progress
)
# Fit GridSearchCV
grid_search.fit(X_train, y_train)

In [ ]:
# Best parameters and accuracy
print(f"Best Parameters: {grid_search.best_params_}")
model_LGB_tuned_one = grid_search.best_estimator_

##### ***Generalized Model***

In [ ]:
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_one_reg, ran_state_given=4)
# choose a model
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model=model_LGB_tuned_one)
evaluate_model(train_preds, y_train, test_preds, y_test)

In [ ]:
plot_learning_curve(model_LGB_tuned_one, X_train, y_train, cv=10, scoring='accuracy')

##### ***Cross-validation***

In [ ]:
random_num = np.random.choice(np.arange(1, 201))
stratified_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_num)
# Define metrics for evaluation
metrics = {
           'accuracy': 'accuracy',
           'f1_score': make_scorer(f1_score),
           'roc_auc': 'roc_auc',
           'precision': make_scorer(precision_score),
           'recall': make_scorer(recall_score)
          }
# Perform cross-validation for the above metrics
cv_results = {}
for metric_name, metric in metrics.items():
  scores = cross_val_score(model_LGB_tuned_one, X, y, cv=stratified_kf, scoring=metric, n_jobs=-1)
  cv_results[metric_name] = scores
  print(f"{metric_name.capitalize()} Scores: {scores}")
  print(f"Mean {metric_name.capitalize()}: {scores.mean():.4f}")
  print(f"Standard Deviation: {scores.std():.4f}\n")

##### ***Feature Importance & Selection***

In [ ]:
# Extract feature importance
feature_importances = model_LGB_tuned_one.feature_importances_
feature_names = X_train.columns if hasattr(X_train, 'columns') else [f"Feature {i}" for i in range(X_train.shape[1])]

# Create a DataFrame for visualization
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

# Display feature importance
print(importance_df)

In [ ]:
df_one_reg_lgb_fea = df_one_reg.drop(['cabin_level', 'traveller_Couple Leisure', 'traveller_Family Leisure', 'traveller_Solo Leisure', 'entertainment'], axis=1)
df_one_reg_lgb_fea.head()

##### ***Final Model for LightGBM Classifier for df_one_reg***

In [ ]:
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_one_reg_lgb_fea, ran_state_given=877)
# creating a Random Forest Classifier Model
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model=model_LGB_tuned_one)
train_accu, test_accu, recall = evaluate_model(train_preds, y_train, test_preds, y_test, evaluation_return = True)

In [ ]:
stratified_cv = StratifiedKFold(n_splits=7, shuffle=True, random_state=88)
plot_learning_curve(model_LGB_tuned_one, X_train, y_train, cv=stratified_cv, scoring='accuracy')

In [ ]:
stratified_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=22)
mean_accu = 0
mean_recall = 0
mean_roc_auc = 0
# Define metrics for evaluation
metrics = {
           'accuracy': 'accuracy',
           'f1_score': make_scorer(f1_score),
           'roc_auc': 'roc_auc',
           'precision': make_scorer(precision_score),
           'recall': make_scorer(recall_score)
          }
# Perform cross-validation for the above metrics
cv_results = {}
for metric_name, metric in metrics.items():
  scores = cross_val_score(model_LGB_tuned_one, X, y, cv=stratified_kf, scoring=metric, n_jobs=-1)
  cv_results[metric_name] = scores
  print(f"{metric_name.capitalize()} Scores: {scores}")
  print(f"Mean {metric_name.capitalize()}: {scores.mean():.4f}")
  print(f"Standard Deviation: {scores.std():.4f}\n")
  if metric_name == 'accuracy':
    mean_accu = scores.mean()
  if metric_name == 'roc_auc':
    mean_roc_auc = scores.mean()
  if metric_name == 'recall':
    mean_recall = scores.mean()

#### ***Final scores for model 7***

In [ ]:
scores_df.loc[6] = ['LightGBM Classifier: df_one_reg', train_accu, mean_accu, mean_roc_auc, mean_recall]
scores_df

#### **LightGBM Classifier: Model 8(df_pmf_reg)**

##### ***Data Splitting and Model Selection***

In [ ]:
df_pmf_reg.head()

In [ ]:
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_pmf_reg, ran_state_given=229)
# choose a model
model_lgb = LGBMClassifier(metric="auc")
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model = model_lgb)
evaluate_model(train_preds, y_train, test_preds, y_test)

In [ ]:
roc_auc_curve(y_test, test_preds, "LightGBM(df_one_reg)")

##### ***Learning Curves***

In [ ]:
plot_learning_curve(model_lgb, X_train, y_train, cv=5, scoring='accuracy')

In [ ]:
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=25)
plot_learning_curve(model_lgb, X_train, y_train, cv=stratified_cv, scoring='accuracy')

##### ***Hyper-parameter Tunning***

In [ ]:
param_grid = {
              'num_leaves': [31, 50],
              'max_depth': [-1, 10],
              'learning_rate': [0.05, 0.1],
              'n_estimators': [100, 200],
              'min_child_samples': [20, 30]
             }

In [ ]:
lgbm = LGBMClassifier()
grid_search = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid,
    cv=3,               # 3-fold cross-validation
    scoring='accuracy', # Optimize for accuracy
    n_jobs=-1,          # Use all available CPU cores
    verbose=1           # Print progress
)
# Fit GridSearchCV
grid_search.fit(X_train, y_train)

In [ ]:
# Best parameters and accuracy
print(f"Best Parameters: {grid_search.best_params_}")
model_LGB_tuned_pmf = grid_search.best_estimator_

##### ***Generalized Model***

In [ ]:
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_pmf_reg, ran_state_given=40)
# choose a model
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model=model_LGB_tuned_pmf)
evaluate_model(train_preds, y_train, test_preds, y_test)

In [ ]:
plot_learning_curve(model_LGB_tuned_pmf, X_train, y_train, cv=10, scoring='accuracy')

##### ***Cross-validation***

In [ ]:
random_num = np.random.choice(np.arange(1, 201))
stratified_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_num)
# Define metrics for evaluation
metrics = {
           'accuracy': 'accuracy',
           'f1_score': make_scorer(f1_score),
           'roc_auc': 'roc_auc',
           'precision': make_scorer(precision_score),
           'recall': make_scorer(recall_score)
          }
# Perform cross-validation for the above metrics
cv_results = {}
for metric_name, metric in metrics.items():
  scores = cross_val_score(model_LGB_tuned_pmf, X, y, cv=stratified_kf, scoring=metric, n_jobs=-1)
  cv_results[metric_name] = scores
  print(f"{metric_name.capitalize()} Scores: {scores}")
  print(f"Mean {metric_name.capitalize()}: {scores.mean():.4f}")
  print(f"Standard Deviation: {scores.std():.4f}\n")

##### ***Feature Importance & Selection***

In [ ]:
# Extract feature importance
feature_importances = model_LGB_tuned_pmf.feature_importances_
feature_names = X_train.columns if hasattr(X_train, 'columns') else [f"Feature {i}" for i in range(X_train.shape[1])]

# Create a DataFrame for visualization
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

# Display feature importance
print(importance_df)

In [ ]:
df_pmf_reg_lgb_fea = df_pmf_reg.drop(['cabin_level', 'traveller_Couple Leisure', 'traveller_Family Leisure', 'traveller_Solo Leisure', 'seat_comfort'], axis=1)
df_pmf_reg_lgb_fea.head()

##### ***Final Model for LightGBM Classifier for df_pmf_reg***

In [ ]:
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_pmf_reg_lgb_fea, ran_state_given=14)
# creating a Random Forest Classifier Model
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model=model_LGB_tuned_pmf)
train_accu, test_accu, recall = evaluate_model(train_preds, y_train, test_preds, y_test, evaluation_return = True)

In [ ]:
stratified_cv = StratifiedKFold(n_splits=7, shuffle=True, random_state=88)
plot_learning_curve(model_LGB_tuned_pmf, X_train, y_train, cv=stratified_cv, scoring='accuracy')

In [ ]:
stratified_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=22)
mean_accu = 0
mean_recall = 0
mean_roc_auc = 0
# Define metrics for evaluation
metrics = {
           'accuracy': 'accuracy',
           'f1_score': make_scorer(f1_score),
           'roc_auc': 'roc_auc',
           'precision': make_scorer(precision_score),
           'recall': make_scorer(recall_score)
          }
# Perform cross-validation for the above metrics
cv_results = {}
for metric_name, metric in metrics.items():
  scores = cross_val_score(model_LGB_tuned_pmf, X, y, cv=stratified_kf, scoring=metric, n_jobs=-1)
  cv_results[metric_name] = scores
  print(f"{metric_name.capitalize()} Scores: {scores}")
  print(f"Mean {metric_name.capitalize()}: {scores.mean():.4f}")
  print(f"Standard Deviation: {scores.std():.4f}\n")
  if metric_name == 'accuracy':
    mean_accu = scores.mean()
  if metric_name == 'roc_auc':
    mean_roc_auc = scores.mean()
  if metric_name == 'recall':
    mean_recall = scores.mean()

#### ***Final scores for model 8***

In [ ]:
scores_df.loc[7] = ['LightGBM Classifier: df_pmf_reg', train_accu, mean_accu, mean_roc_auc, mean_recall]
scores_df

#### **LightGBM Classifier: Model 9(df_hug_reg)**

##### ***Data Splitting and Model Selection***

In [ ]:
df_hug_reg.head()

In [ ]:
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_hug_reg, ran_state_given=29)
# choose a model
model_lgb = LGBMClassifier(metric="auc")
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model = model_lgb)
evaluate_model(train_preds, y_train, test_preds, y_test)

In [ ]:
roc_auc_curve(y_test, test_preds, "LightGBM(df_one_reg)")

##### ***Learning Curves***

In [ ]:
plot_learning_curve(model_lgb, X_train, y_train, cv=5, scoring='accuracy')

In [ ]:
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=25)
plot_learning_curve(model_lgb, X_train, y_train, cv=stratified_cv, scoring='accuracy')

##### ***Hyper-parameter Tunning***

In [ ]:
param_dist = {
    'num_leaves': np.arange(10, 50, 5),              # Smaller range for small datasets
    'max_depth': [-1, 5, 10, 15],                   # Control tree depth
    'learning_rate': [0.01, 0.05, 0.1],             # Learning rate
    'n_estimators': np.arange(50, 200, 25),         # Fewer boosting iterations
    'min_child_samples': np.arange(5, 20, 2),       # Ensure enough samples in leaf nodes
    'subsample': [0.6, 0.8, 1.0],                   # Fraction of data per tree
    'colsample_bytree': [0.6, 0.8, 1.0],            # Fraction of features per tree
    'reg_alpha': [0, 0.1, 1],                       # L1 regularization
    'reg_lambda': [0, 0.1, 1]                       # L2 regularization
}

In [ ]:
lgbm = LGBMClassifier()
# RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=param_dist,
    n_iter=30,                  # Try 30 random combinations
    scoring='accuracy',         # Optimize for accuracy
    cv=5,                       # Use 5-fold cross-validation
    verbose=2,                  # Print progress
    random_state=42,            # Ensure reproducibility
    n_jobs=-1                   # Use all available CPU cores
)
# Fit GridSearchCV
random_search.fit(X, y)

In [ ]:
# Best parameters and accuracy
print(f"Best Parameters: {random_search.best_params_}")
model_LGB_tuned_hug = random_search.best_estimator_

##### ***Generalized Model***

In [ ]:
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_hug_reg, ran_state_given=42)
# choose a model
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model=model_LGB_tuned_hug)
evaluate_model(train_preds, y_train, test_preds, y_test)

In [ ]:
plot_learning_curve(model_LGB_tuned_hug, X_train, y_train, cv=10, scoring='accuracy')

##### ***Cross-validation***

In [ ]:
random_num = np.random.choice(np.arange(1, 201))
stratified_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_num)
# Define metrics for evaluation
metrics = {
           'accuracy': 'accuracy',
           'f1_score': make_scorer(f1_score),
           'roc_auc': 'roc_auc',
           'precision': make_scorer(precision_score),
           'recall': make_scorer(recall_score)
          }
# Perform cross-validation for the above metrics
cv_results = {}
for metric_name, metric in metrics.items():
  scores = cross_val_score(model_LGB_tuned_hug, X, y, cv=stratified_kf, scoring=metric, n_jobs=-1)
  cv_results[metric_name] = scores
  print(f"{metric_name.capitalize()} Scores: {scores}")
  print(f"Mean {metric_name.capitalize()}: {scores.mean():.4f}")
  print(f"Standard Deviation: {scores.std():.4f}\n")

##### ***Feature Importance & Selection***

In [ ]:
# Extract feature importance
feature_importances = model_LGB_tuned_hug.feature_importances_
feature_names = X_train.columns if hasattr(X_train, 'columns') else [f"Feature {i}" for i in range(X_train.shape[1])]

# Create a DataFrame for visualization
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

# Display feature importance
print(importance_df)

In [ ]:
df_hug_reg_lgb_fea = df_hug_reg.drop(['cabin_level', 'traveller_Couple Leisure', 'traveller_Family Leisure', 'traveller_Solo Leisure', 'food_bev', 'entertainment', 'food_bev'], axis=1)
df_hug_reg_lgb_fea.head()

##### ***Final Model for LightGBM Classifier for df_pmf_reg***

In [ ]:
# split dataset into feature and target datasets
X, y, X_train, X_test, y_train, y_test = split_data_train_test(df=df_hug_reg_lgb_fea, ran_state_given=14)
# creating a Random Forest Classifier Model
train_preds, test_preds = fit_model(X_train, y_train, X_test, y_test, model=model_LGB_tuned_hug)
train_accu, test_accu, recall = evaluate_model(train_preds, y_train, test_preds, y_test, evaluation_return = True)

In [ ]:
stratified_cv = StratifiedKFold(n_splits=15, shuffle=True, random_state=88)
plot_learning_curve(model_LGB_tuned_hug, X_train, y_train, cv=stratified_cv, scoring='accuracy')

In [ ]:
stratified_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=22)
mean_accu = 0
mean_recall = 0
mean_roc_auc = 0
# Define metrics for evaluation
metrics = {
           'accuracy': 'accuracy',
           'f1_score': make_scorer(f1_score),
           'roc_auc': 'roc_auc',
           'precision': make_scorer(precision_score),
           'recall': make_scorer(recall_score)
          }
# Perform cross-validation for the above metrics
cv_results = {}
for metric_name, metric in metrics.items():
  scores = cross_val_score(model_LGB_tuned_hug, X, y, cv=stratified_kf, scoring=metric, n_jobs=-1)
  cv_results[metric_name] = scores
  print(f"{metric_name.capitalize()} Scores: {scores}")
  print(f"Mean {metric_name.capitalize()}: {scores.mean():.4f}")
  print(f"Standard Deviation: {scores.std():.4f}\n")
  if metric_name == 'accuracy':
    mean_accu = scores.mean()
  if metric_name == 'roc_auc':
    mean_roc_auc = scores.mean()
  if metric_name == 'recall':
    mean_recall = scores.mean()

#### ***Final scores for model 9***

In [ ]:
scores_df.loc[8] = ['LightGBM Classifier: df_pmf_hug', train_accu, mean_accu, mean_roc_auc, mean_recall]
scores_df

#### Explain the ML Model used and it's performance using Evaluation metric Score Chart.

For our 3rd model, we selected LightGBM, another tree based model for classification:

- **Data Preparation**: The datasets are split into training and testing sets with a specific random state.
- **Initial Model:** A baseline LGBMClassifier is created with "auc" as the optimization metric.
- **Hyperparameter Optimization:** GridSearchCV is implemented with:
Parameter grid exploring: num_leaves, max_depth, learning_rate, n_estimators, and min_child_samples
- 3-fold cross-validation
- Accuracy as the scoring metric
- Parallel processing (n_jobs=-1)
- **Model Finalization**: The best model from GridSearchCV is extracted and evaluated on a new data split.
- **Robust Evaluation:** The model undergoes 5-fold stratified cross-validation with multiple metrics: accuracy, F1 score, ROC-AUC, precision, and recall.

**Analysis and Observations:**

- **Exceptional Stability:** The extremely low standard deviations across all metrics indicate a highly robust model that generalizes well across different data samples.
- **Superior ROC-AUC:** The near-perfect ROC-AUC score of 0.9922 demonstrates excellent discriminative ability between classes.
- **Balanced Performance**: The model maintains a good balance between precision and recall, suggesting it handles both classes effectively without significant bias.

#### Which hyperparameter optimization technique have you used and why?

We used GridSearchCV for hyperparameter optimization. This approach systematically evaluates all possible combinations of parameters from the provided grid:

- The GridSearchCV was configured with:

  - 3-fold cross-validation
  - Accuracy as the optimization metric
  - Parallel processing to utilize all CPU cores

- **Comprehensive Exploration:** GridSearchCV exhaustively tests all parameter combinations, ensuring the optimal configuration isn't missed.
- **Parameter Interactions:** It captures how parameters interact with each other, which is crucial for tree-based models like LightGBM where parameters often have interdependent effects.
- **Cross-Validation Integration:** The built-in cross-validation ensures the selected parameters generalize well rather than overfitting to a specific data split.
- **Appropriate Scale:** Parameter grid is focused and reasonable in size (32 total combinations), making GridSearchCV an efficient choice without being computationally prohibitive.
- **Objective Alignment:** By using accuracy as the scoring metric, we've aligned the optimization directly with one of your primary evaluation metrics.

The excellent results with very low standard deviations across our cross-validation metrics validate this choice, demonstrating that GridSearchCV effectively identified parameters that produce both high performance and remarkable stability.

#### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

Again, the PMF-based model shows improvements across all metrics, with the **most significant gains in F1 Score and Accuracy**. The **df_pmf_reg model** has achieved the highest rank among the tested models, confirming that the **probability mass function** approach creates features that allow LightGBM to make more accurate and reliable predictions.
Interestingly, while the **df_pmf_hug model has the highest F1 Score**, its precision is lower than both other models, this may be due to the low training data given to this model(~5,000). **This suggests that the df_pmf_reg model provides the best overall balance of performance metrics.**

### **Final Score of all models**

In [ ]:
scores_df

In [ ]:
scores_df.sort_values(by="Accuracy(after CV)", ascending=False)

## ***7. Model Explanation***

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

Our  main objective is to predict whether passengers will refer the airline to their friends.

- **Dataset Impact:**
  The PMF-based dataset consistently delivers superior performance across all algorithms, suggesting that probability-based feature engineering significantly enhances prediction quality.

- **Algorithm Performance:**
LightGBM demonstrates the strongest overall performance, particularly when combined with the PMF-based features, achieving both the highest accuracy and excellent recall.

- **Recall Excellence:**
All models show outstanding recall (>98.5%), indicating strong capability in identifying passengers likely to recommend the airline. This is crucial as failing to identify potential promoters represents a missed business opportunity.

- **Business Implications**
  - The top-performing model can identify nearly 99.5% of passengers who would recommend the airline, enabling targeted marketing strategies to leverage these potential brand advocates.
  - High precision (96.77% for the best model) ensures that resources spent on predicted promoters are efficiently allocated, minimizing wasteful investments in passengers unlikely to recommend.
  - The superior performance of PMF-based models demonstrates the value of transforming raw features into probability distributions that better represent passenger behavior patterns.
  - Even the lower-ranked models show solid performance, indicating that the predictive patterns are robust across different modeling approaches.

**Conclusion:**

The LightGBM Classifier & Logistic Regression Models trained on PMF-transformed data provides the optimal solution for our recommendation prediction task, balancing high recall (capturing potential promoters) with strong precision (accuracy in positive predictions). The minimal performance difference between the top three models suggests that the PMF transformation is the critical factor in achieving superior results, regardless of the specific algorithm employed.

**Note(s):**

It can be said that the hugging face model trained on a very small dataset(~5,000) datapoints has shown the highest recall, since our main objective is to identify customers who would recommend the airlines to other, this model if trained by manipulating the dataset by creating sentiment varibales combined with pmf imputer might have the strongest impact. Due to limited resouces, this model could not be explored further!

- The Logistic Regression model trained on the hugging face dataset (df_hug_reg) achieved an F1 score of 0.975677, which is actually the highest among all models tested. This is particularly impressive considering it was trained on only about 5,000 data points compared to the much larger datasets used for other models.
- This suggests that the sentiment-based features extracted by the hugging face model are capturing crucial aspects of customer satisfaction that strongly correlate with recommendation likelihood. The high recall indicates that this approach is especially effective at identifying potential brand advocates.
- Our hypothesis about combining the hugging face sentiment variables with the PMF imputer technique is particularly insightful. Such a hybrid approach could potentially leverage:

  - The nuanced sentiment understanding from the language model
  - The statistical robabilities captured by the PMF transformation
  - The larger dataset to improve generalization

This combination could indeed create a breakthrough model with even stronger predictive power. The resource constraints we mentioned are a common limitation in model exploration, but this definitely identifies a promising direction for future research if resources become available.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

As for our final model, we can choose any between the **LightGBM Classifier: df_pmf_reg** and **Logistic Regression: df_pmf_reg**. One has a higher accuracy, while the other has a higher recall. **Ultimately, we have decided to choose the Logistic Regression: df_pmf_reg model**, since the accuracy difference between both is very minute, but the ROC AUC score and Recall of the Logistics Regression model is minutely better, additionally training the logistic model is much more time and cost efficient which in our case suits with the objective of this project.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

We will use the Logistics Regression PMF model as our production model. Using SHAP and feature importance feature for the model, we will be using the following variables:
- overall         
- ground_service         
- value_for_money         
- cabin_service        
- entertainment        
- food_bev

for such, we will need to modify our inputs, or adjust our dataset so that it may only have the following variables.

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
# Save the File
!pip install joblib
import joblib

In [ ]:
#Random Forest Regressor model
prod_model = model_LoR_pmf
# Specify the file path to save the model
model_filename = 'prod_model_airline_ref.joblib'
# Save the model to the file
joblib.dump(prod_model, model_filename)

print(f"Model saved as {model_filename}")

### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Load the File and predict unseen data.
model_filename = 'prod_model_airline_ref.joblib'
loaded_model = joblib.load(model_filename)

# Create some sample unseen data
# 'overall', 'cabin_service', 'food_bev', 'entertainment', 'ground_service', 'value_for_money'
X_unseen = np.array([[0.666667, 1.00, 0.75, 0.75, 0.25, 0.75],  # recommendation = 1(yes)
                     [0.222222,	0.75,	0.00,	0.50,	0.00,	0.25]]) # recommendation = 0(no)

# Make predictions using the loaded model
predictions = loaded_model.predict(X_unseen)

print("Loaded model:", loaded_model)
print("Sample prediction for unseen data:", predictions)

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

Our objective was to build a machine learning model that will successfully predict whether a customer would recommend an airline based on their experience.

**Task: Binary Classification**

**Data Processing and Exploration**
- Downloaded the dataset from local Google Drive, performed basic data wrangling operations, and cleaned up the dataset
- Explored features and target variables to understand the data structure
- Performed comprehensive EDA, visualizing relationships between variables through 15 charts including heatmaps and correlation graphs

**Dataset Preparation**
- Created three distinct datasets for modeling:
  - Regular: Standard dataset with simple imputation (mean and median) for missing values
  - PMF: Probability Mass Function dataset with PMF-based imputation for missing values
  - Hugg: Dataset with all rows containing missing values removed, enhanced with three new feature variables from ROBERTA sentiment analysis

**Model Development and Optimization**
- Implemented 3 machine learning algorithms across all 3 datasets (total of 9 models):
  - Logistic Regression
  - Random Forest Classifier
  - LightGBM

- For each model:
  - Fitted with 80:20 train-test split and Min-Max scaling
  - Evaluated with focus on precision and recall metrics
  - Plotted learning curves to diagnose overfitting/underfitting
  - Tuned parameters when overfitting was detected to minimize train-test accuracy gap
  - Performed hyperparameter optimization on promising models
  - Analyzed feature importance and removed 4-5 least important features
  - Created and fitted final optimized models

**Deployment**
- Saved the best performing model in joblib format
- Validated model performance on unseen test data

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***